# Long Jump Performance Analysis

Bachelor's Thesis analysis notebook. The workflow converts video-derived biomechanical features into a clean attempt-level dataset and evaluates three machine-learning tasks: valid/foul classification, absolute jump-distance regression, and performance relative to each athlete's previous level.

The raw competition footage is not included in this repository.


## 1. Setup and paths

Imports, directory configuration and project paths.


In [ ]:
from pathlib import Path
import warnings
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 250)


BASE_DIR_MANUAL = None


VIDEOS_A_REVISAR_MAS_ADELANTE = [
    "Longi_MASC_070",
    "Longi_MASC_052",
]


VIDEOS_EXCLUIR_MANUAL = []

EVENTOS_USAR = [
    "apoyo_-4",
    "apoyo_-3",
    "apoyo_-2",
    "apoyo_-1",
    "batida",
]

PATRONES_CONTROL_GENERAL = [
    "frame",
    "score",
    "estado",
    "criterio",
    "pose",
    "crop",
    "pixel",
    "_px",
    "x_original",
    "y_original",
    "_x_",
    "_y_",
    "vis",
    "fuente",
    "offset",
    "debug",
    "archivo_origen",
    "metodo_recuperacion",
    "distancia_frame",
]


In [ ]:
def encontrar_results_dir():
    if BASE_DIR_MANUAL is not None:
        base = Path(BASE_DIR_MANUAL)
        return base, base / "resultados_revision_individual"

    posibles = [
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "Desktop" / "PROYECT",
        Path.home() / "Escritorio" / "PROYECT",
        Path.home() / "Desktop" / "PROJECT",
        Path.home() / "Escritorio" / "PROJECT",
    ]

    for base in posibles:
        results = base / "resultados_revision_individual"
        if results.exists():
            return base, results

    for root in [Path.cwd(), Path.home() / "Desktop", Path.home() / "Escritorio"]:
        if root.exists():
            try:
                encontrados = list(root.rglob("resultados_revision_individual"))
                if encontrados:
                    return encontrados[0].parent, encontrados[0]
            except Exception:
                pass

    return Path.cwd(), Path.cwd() / "resultados_revision_individual"


BASE_DIR, RESULTS_DIR = encontrar_results_dir()

ENRICHED_DIR = RESULTS_DIR / "eventos_enriquecidos_apoyo_aire"
ENRICHED_EVENTS_DIR = ENRICHED_DIR / "eventos_por_video"

OUT_DIR = RESULTS_DIR / "eda_por_bloques_apoyo_aire_tfg_v2"
DATASETS_DIR = OUT_DIR / "datasets"
TABLAS_DIR = OUT_DIR / "tablas"
FIG_DIR = OUT_DIR / "figuras"

for d in [OUT_DIR, DATASETS_DIR, TABLAS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("Existe RESULTS_DIR:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR)
print("Existe ENRICHED_DIR:", ENRICHED_DIR.exists())
print("OUT_DIR:", OUT_DIR)
print("Vídeos a revisar más adelante, pero NO excluidos ahora:", VIDEOS_A_REVISAR_MAS_ADELANTE)


## 2. Shared utilities

Identifier normalization, quality summaries, missing-value handling and reusable dataset helpers.


In [ ]:
def normalizar_video_id(valor):
    if pd.isna(valor):
        return np.nan

    s = str(valor)
    s = Path(s).stem
    s = s.replace("eventos_automaticos_", "")
    s = s.replace("eventos_enriquecidos_", "")
    s = s.replace("variables_velocidad_batida_", "")
    s = s.replace("keypoints_angulos_", "")

    m = re.search(r"(Longi_[A-Za-z]+_\d+)", s)
    if m:
        return m.group(1)

    return s


def limpiar_nombre_evento(evento):
    return (
        str(evento)
        .replace("apoyo_-", "apoyo_m")
        .replace("-", "m")
        .replace(" ", "_")
    )


def asegurar_video_id_primera_columna(df):
    df = df.copy()

    if "video_id" not in df.columns:
        raise ValueError("El dataframe no tiene video_id. No se puede guardar sin referencia de vídeo.")

    df["video_id"] = df["video_id"].apply(normalizar_video_id)

    cols = ["video_id"] + [c for c in df.columns if c != "video_id"]
    return df[cols]


def contiene_patron_control(col, patrones=PATRONES_CONTROL_GENERAL):
    c = str(col).lower()
    return any(p in c for p in patrones)


def guardar_csv(df, path):
    df = asegurar_video_id_primera_columna(df)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def resumen_calidad_dataset(df, nombre_dataset):
    df = asegurar_video_id_primera_columna(df)

    filas = []

    for col in df.columns:
        serie = df[col]

        fila = {
            "dataset": nombre_dataset,
            "variable": col,
            "tipo": str(serie.dtype),
            "n_nulos": int(serie.isna().sum()),
            "pct_nulos": float(serie.isna().mean() * 100),
            "n_unicos": int(serie.nunique(dropna=True)),
        }

        if pd.api.types.is_numeric_dtype(serie):
            s = pd.to_numeric(serie, errors="coerce")
            fila.update({
                "min": s.min(),
                "max": s.max(),
                "media": s.mean(),
                "mediana": s.median(),
                "std": s.std(),
            })
        else:
            fila.update({
                "min": np.nan,
                "max": np.nan,
                "media": np.nan,
                "mediana": np.nan,
                "std": np.nan,
            })

        filas.append(fila)

    return pd.DataFrame(filas)


def imputar_mediana(df):
    out = asegurar_video_id_primera_columna(df)
    resumen = []

    for col in out.columns:
        if col == "video_id":
            continue

        if pd.api.types.is_numeric_dtype(out[col]):
            n_missing = int(out[col].isna().sum())
            med = out[col].median()

            if n_missing > 0:
                out[col] = out[col].fillna(med)

            resumen.append({
                "variable": col,
                "n_imputados": n_missing,
                "pct_imputado": n_missing / len(out) * 100 if len(out) else np.nan,
                "mediana_usada": med,
            })

    return out, pd.DataFrame(resumen)


def guardar_dataset_con_imputacion(df, nombre):
    df = asegurar_video_id_primera_columna(df)

    path_sin = DATASETS_DIR / f"{nombre}_sin_imputar.csv"
    guardar_csv(df, path_sin)

    df_imp, resumen_imp = imputar_mediana(df)
    path_imp = DATASETS_DIR / f"{nombre}_imputado_mediana.csv"
    guardar_csv(df_imp, path_imp)

    resumen_imp.to_csv(TABLAS_DIR / f"imputacion_{nombre}.csv", index=False, encoding="utf-8-sig")

    print(f"{nombre}: {df.shape} -> {path_sin.name} / {path_imp.name}")
    return df_imp


def graficar_nulos(resumen, nombre):
    if resumen.empty:
        return

    tmp = resumen[resumen["variable"] != "video_id"].copy()
    tmp = tmp.sort_values("pct_nulos", ascending=False).head(30)

    if tmp.empty:
        return

    tmp = tmp.sort_values("pct_nulos")

    plt.figure(figsize=(10, max(5, 0.30 * len(tmp))))
    plt.barh(tmp["variable"], tmp["pct_nulos"])
    plt.xlabel("% de nulos")
    plt.title(f"Nulos por variable - {nombre}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"nulos_{nombre}.png", dpi=300, bbox_inches="tight")
    plt.show()


def analizar_y_guardar(df, nombre):
    if df is None or df.empty:
        print(f"{nombre}: dataset vacío")
        return pd.DataFrame()

    df = asegurar_video_id_primera_columna(df)
    resumen = resumen_calidad_dataset(df, nombre)
    resumen.to_csv(TABLAS_DIR / f"calidad_{nombre}.csv", index=False, encoding="utf-8-sig")
    graficar_nulos(resumen, nombre)
    guardar_dataset_con_imputacion(df, nombre)
    return resumen


def quitar_columnas_totalmente_vacias(df):
    if df is None or df.empty:
        return df
    if "video_id" not in df.columns:
        return df.dropna(axis=1, how="all")
    video_id = df["video_id"]
    resto = df.drop(columns=["video_id"]).dropna(axis=1, how="all")
    out = pd.concat([video_id, resto], axis=1)
    return asegurar_video_id_primera_columna(out)


def merge_por_video(dfs):
    dfs = [asegurar_video_id_primera_columna(df) for df in dfs if df is not None and not df.empty]

    if not dfs:
        return pd.DataFrame()

    out = dfs[0].copy()

    for df_next in dfs[1:]:
        out = out.merge(df_next, on="video_id", how="inner")

    return asegurar_video_id_primera_columna(out)


## 3. Build biomechanical feature blocks

The extracted event data are reshaped to one row per attempt. Entry velocity and common-frame post-take-off velocity are added as separate feature blocks.


In [ ]:
def cargar_eventos_enriquecidos():
    path_todos = ENRICHED_DIR / "eventos_enriquecidos_todos.csv"

    if path_todos.exists():
        df_ev = pd.read_csv(path_todos)
        print("Cargado:", path_todos)
    else:
        files = sorted(ENRICHED_EVENTS_DIR.glob("eventos_enriquecidos_*.csv"))
        if not files:
            raise FileNotFoundError(
                "No encuentro eventos enriquecidos. Ejecuta primero enriquecer_eventos_con_keypoints.py"
            )

        dfs = []
        for f in files:
            tmp = pd.read_csv(f)
            if "video_id" not in tmp.columns:
                tmp["video_id"] = normalizar_video_id(f.name)
            dfs.append(tmp)

        df_ev = pd.concat(dfs, ignore_index=True)
        print("Cargados eventos por vídeo:", len(files))

    if "video_id" not in df_ev.columns:
        if "video" in df_ev.columns:
            df_ev["video_id"] = df_ev["video"].apply(normalizar_video_id)
        else:
            raise ValueError("No encuentro columna video_id ni video en eventos enriquecidos.")

    df_ev["video_id"] = df_ev["video_id"].apply(normalizar_video_id)
    df_ev = asegurar_video_id_primera_columna(df_ev)

    return df_ev


eventos = cargar_eventos_enriquecidos()


eventos = eventos[
    ~eventos["video_id"].isin(VIDEOS_EXCLUIR_MANUAL)
].copy()

print("Eventos enriquecidos:", eventos.shape)
display(eventos.head())
display(eventos.columns.tolist())


In [ ]:
VARIABLES_EVENTOS_TECNICAS = [
    "distancia_a_tabla_m",
    "longitud_hasta_siguiente_m",
    "tiempo_hasta_siguiente_s",
    "velocidad_paso_m_s",

    "angulo_rodilla_apoyo",
    "angulo_cadera_apoyo",
    "angulo_rodilla_aire",
    "angulo_cadera_aire",
    "dif_rodilla_apoyo_aire",
    "dif_cadera_apoyo_aire",

    "inclinacion_tronco",
]


if "angulo_rodilla_apoyo" not in eventos.columns and "angulo_rodilla" in eventos.columns:
    eventos["angulo_rodilla_apoyo"] = eventos["angulo_rodilla"]

if "angulo_cadera_apoyo" not in eventos.columns and "angulo_cadera" in eventos.columns:
    eventos["angulo_cadera_apoyo"] = eventos["angulo_cadera"]

VARIABLES_EVENTOS_TECNICAS = [c for c in VARIABLES_EVENTOS_TECNICAS if c in eventos.columns]

eventos_limpios = eventos[
    ["video_id", "evento"] + VARIABLES_EVENTOS_TECNICAS
].copy()

eventos_limpios = asegurar_video_id_primera_columna(eventos_limpios)


eventos_limpios.to_csv(
    DATASETS_DIR / "00_eventos_limpios_largo.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Eventos limpios largos:", eventos_limpios.shape)
display(eventos_limpios.head(15))


In [ ]:
eventos_det = eventos_limpios[
    eventos_limpios["evento"].isin(EVENTOS_USAR)
].copy()

eventos_det["evento_limpio"] = eventos_det["evento"].apply(limpiar_nombre_evento)

piezas = []

for var in VARIABLES_EVENTOS_TECNICAS:
    pivot = eventos_det.pivot_table(
        index="video_id",
        columns="evento_limpio",
        values=var,
        aggfunc="first"
    )

    pivot.columns = [f"{evento}_{var}" for evento in pivot.columns]
    piezas.append(pivot)

dataset_eventos_detallado = pd.concat(piezas, axis=1).reset_index() if piezas else pd.DataFrame()
dataset_eventos_detallado = quitar_columnas_totalmente_vacias(dataset_eventos_detallado)


columnas_quitar_batida_distancia = [
    c for c in dataset_eventos_detallado.columns
    if c.lower().startswith("batida_")
    and "distancia_a_tabla" in c.lower()
]

dataset_eventos_detallado = dataset_eventos_detallado.drop(
    columns=columnas_quitar_batida_distancia,
    errors="ignore"
)

dataset_eventos_detallado = asegurar_video_id_primera_columna(dataset_eventos_detallado)

print("Columnas eliminadas por distancia a tabla en batida:")
for c in columnas_quitar_batida_distancia:
    print("-", c)

print("Dataset eventos detallado:", dataset_eventos_detallado.shape)
display(dataset_eventos_detallado.head())


dataset_eventos_resumen = pd.DataFrame()

print("No se crea dataset_eventos_resumen. Se conserva únicamente dataset_eventos_detallado.")


In [ ]:
def cargar_velocidades_normales():
    files = sorted(RESULTS_DIR.glob("variables_velocidad_batida_*.csv"))

    if not files:
        print("No se encontraron variables_velocidad_batida_*.csv")
        return pd.DataFrame()

    dfs = []

    for f in files:
        tmp = pd.read_csv(f)

        if "video_id" not in tmp.columns:
            if "video" in tmp.columns:
                tmp["video_id"] = tmp["video"].apply(normalizar_video_id)
            else:
                tmp["video_id"] = normalizar_video_id(f.name)

        tmp["video_id"] = tmp["video_id"].apply(normalizar_video_id)
        tmp["archivo_origen"] = f.name
        dfs.append(tmp)

    out = pd.concat(dfs, ignore_index=True)


    out = out.drop_duplicates("video_id")
    out = asegurar_video_id_primera_columna(out)

    return out


velocidades_raw = cargar_velocidades_normales()


if not velocidades_raw.empty:
    velocidades_raw = velocidades_raw[
        ~velocidades_raw["video_id"].isin(VIDEOS_EXCLUIR_MANUAL)
    ].copy()
    velocidades_raw = asegurar_video_id_primera_columna(velocidades_raw)

print("Velocidades normales raw:", velocidades_raw.shape)
display(velocidades_raw.head())
display(velocidades_raw.columns.tolist())


In [ ]:
def es_columna_salida_normal(col):
    """
    Detecta columnas de salida/postbatida del CSV normal de velocidades.
    Se eliminan porque esa información se tomará del archivo de frame común.
    """
    c = str(col).lower()

    patrones_salida = [
        "salida",
        "postbatida",
        "post_batida",
        "perdida",
        "pérdida",
        "ratio_salida",
        "ratio",
        "10f",
        "15f",
        "20f",
        "frame_10",
        "frame_15",
        "frame_20",
        "a_10",
        "a_15",
        "a_20",
    ]

    return any(p in c for p in patrones_salida)


def es_columna_control_velocidad_normal(col):
    """
    Variables de control del CSV normal de velocidad.
    Se conserva video_id siempre.
    """
    if col == "video_id":
        return False

    c = str(col).lower()


    if c == "archivo_origen":
        return True

    patrones_control = [
        "frame",
        "score",
        "estado",
        "criterio",
        "debug",
        "archivo",
        "source",
        "fuente",
        "n_frames",
        "num_frames",
    ]

    return any(p in c for p in patrones_control)


if velocidades_raw.empty:
    dataset_velocidad_normal_sin_salida = pd.DataFrame()
    columnas_eliminadas_velocidad_normal = pd.DataFrame()
else:
    columnas_quitar_salida = [
        c for c in velocidades_raw.columns
        if c != "video_id" and es_columna_salida_normal(c)
    ]

    columnas_quitar_control = [
        c for c in velocidades_raw.columns
        if c != "video_id" and es_columna_control_velocidad_normal(c)
    ]

    columnas_quitar = sorted(set(columnas_quitar_salida + columnas_quitar_control))

    dataset_velocidad_normal_sin_salida = velocidades_raw.drop(
        columns=columnas_quitar,
        errors="ignore"
    ).copy()


    cols_utiles = ["video_id"] + [
        c for c in dataset_velocidad_normal_sin_salida.columns
        if c != "video_id" and pd.api.types.is_numeric_dtype(dataset_velocidad_normal_sin_salida[c])
    ]

    dataset_velocidad_normal_sin_salida = dataset_velocidad_normal_sin_salida[cols_utiles].copy()
    dataset_velocidad_normal_sin_salida = quitar_columnas_totalmente_vacias(dataset_velocidad_normal_sin_salida)
    dataset_velocidad_normal_sin_salida = asegurar_video_id_primera_columna(dataset_velocidad_normal_sin_salida)

    columnas_eliminadas_velocidad_normal = pd.DataFrame(
        [{"variable_eliminada": c,
          "motivo": "salida_postbatida_o_control_en_csv_normal_de_velocidades"}
         for c in columnas_quitar]
    )

columnas_eliminadas_velocidad_normal.to_csv(
    TABLAS_DIR / "variables_eliminadas_csv_velocidad_normal.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Columnas eliminadas del CSV normal de velocidades:")
display(columnas_eliminadas_velocidad_normal)

print("Dataset velocidad normal sin salida:", dataset_velocidad_normal_sin_salida.shape)
display(dataset_velocidad_normal_sin_salida.head())
display(dataset_velocidad_normal_sin_salida.columns.tolist())


In [ ]:
def cargar_velocidad_frame_comun():
    path_frame_comun = RESULTS_DIR / "velocidad_media_batida_a_frame_comun_min40.csv"

    if not path_frame_comun.exists():
        print("No existe:", path_frame_comun)
        return pd.DataFrame()

    df = pd.read_csv(path_frame_comun)

    if "video_id" not in df.columns:
        if "video" in df.columns:
            df["video_id"] = df["video"].apply(normalizar_video_id)
        else:
            raise ValueError("El archivo de frame común no tiene video_id ni video.")

    df["video_id"] = df["video_id"].apply(normalizar_video_id)


    df = df[
        ~df["video_id"].isin(VIDEOS_EXCLUIR_MANUAL)
    ].copy()

    df = asegurar_video_id_primera_columna(df)

    return df


frame_comun_raw = cargar_velocidad_frame_comun()

print("Frame común raw:", frame_comun_raw.shape)
display(frame_comun_raw.head())
display(frame_comun_raw.columns.tolist())


In [ ]:
if frame_comun_raw.empty:
    dataset_velocidad_salida_frame_comun = pd.DataFrame()
    columnas_eliminadas_frame_comun = pd.DataFrame()

else:

    columnas_frame_comun_validas = [
        "video_id",


        "delta_t_velocidad_media_s",


        "delta_x_m",
        "delta_y_m_vertical_positiva_arriba",


        "velocidad_media_horizontal_signed_m_s",
        "velocidad_media_horizontal_abs_m_s",
        "velocidad_media_vertical_m_s",
        "velocidad_media_resultante_m_s",
    ]

    columnas_frame_comun_validas = [
        c for c in columnas_frame_comun_validas
        if c in frame_comun_raw.columns
    ]

    dataset_velocidad_salida_frame_comun = frame_comun_raw[
        columnas_frame_comun_validas
    ].copy()

    dataset_velocidad_salida_frame_comun = quitar_columnas_totalmente_vacias(
        dataset_velocidad_salida_frame_comun
    )

    dataset_velocidad_salida_frame_comun = asegurar_video_id_primera_columna(
        dataset_velocidad_salida_frame_comun
    )


    columnas_eliminadas = [
        c for c in frame_comun_raw.columns
        if c not in columnas_frame_comun_validas
    ]

    columnas_eliminadas_frame_comun = pd.DataFrame(
        [
            {
                "variable_eliminada": c,
                "motivo": "variable_de_pixel_frame_control_o_no_interpretable_para_modelo"
            }
            for c in columnas_eliminadas
        ]
    )

columnas_eliminadas_frame_comun.to_csv(
    TABLAS_DIR / "variables_eliminadas_csv_frame_comun.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Columnas eliminadas de frame común:")
display(columnas_eliminadas_frame_comun)

print("Dataset velocidad salida frame común limpio:", dataset_velocidad_salida_frame_comun.shape)
display(dataset_velocidad_salida_frame_comun.head())
display(dataset_velocidad_salida_frame_comun.columns.tolist())


## 4. Quality control and cleaning

The feature blocks are checked for coverage, missing values and outliers. Problematic attempts are reviewed before producing the final modelling dataset.


In [ ]:
datasets_bloques = {
    "eventos_detallado": dataset_eventos_detallado,
    "velocidad_normal_sin_salida": dataset_velocidad_normal_sin_salida,
    "velocidad_salida_frame_comun": dataset_velocidad_salida_frame_comun,
}

resumenes = []

for nombre, df_bloque in datasets_bloques.items():
    if df_bloque is None or df_bloque.empty:
        print(f"Saltando {nombre}: dataset vacío")
        continue

    df_bloque = asegurar_video_id_primera_columna(df_bloque)
    datasets_bloques[nombre] = df_bloque

    print("\n" + "="*80)
    print(nombre, df_bloque.shape)
    display(df_bloque.head())

    resumen = analizar_y_guardar(df_bloque, nombre)
    resumenes.append(resumen)

resumen_calidad_todos = pd.concat(resumenes, ignore_index=True) if resumenes else pd.DataFrame()
resumen_calidad_todos.to_csv(TABLAS_DIR / "00_calidad_todos_los_bloques.csv", index=False, encoding="utf-8-sig")


In [ ]:
comparacion = []

for nombre, df_bloque in datasets_bloques.items():
    if df_bloque is None or df_bloque.empty:
        continue

    df_bloque = asegurar_video_id_primera_columna(df_bloque)
    vars_modelo = [c for c in df_bloque.columns if c != "video_id"]

    if len(vars_modelo) == 0:
        continue

    nulos = df_bloque[vars_modelo].isna().mean() * 100

    comparacion.append({
        "bloque": nombre,
        "n_videos": df_bloque.shape[0],
        "n_variables": len(vars_modelo),
        "pct_nulos_medio": nulos.mean(),
        "pct_nulos_max": nulos.max(),
        "n_variables_sin_nulos": int((nulos == 0).sum()),
        "n_variables_con_nulos": int((nulos > 0).sum()),
        "n_variables_mas_20pct_nulos": int((nulos > 20).sum()),
    })

comparacion_bloques = pd.DataFrame(comparacion)

if not comparacion_bloques.empty:
    comparacion_bloques = comparacion_bloques.sort_values("pct_nulos_medio")
    display(comparacion_bloques)
    comparacion_bloques.to_csv(TABLAS_DIR / "01_comparacion_bloques.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(9,5))
    plt.bar(comparacion_bloques["bloque"], comparacion_bloques["pct_nulos_medio"])
    plt.ylabel("% medio de nulos")
    plt.title("Comparación de nulos medios por bloque")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "comparacion_nulos_bloques.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No hay bloques para comparar.")


In [ ]:
def detectar_outliers_iqr_y_mad(df, nombre_dataset):
    df = asegurar_video_id_primera_columna(df).copy()

    resultados = []

    columnas_numericas = [
        c for c in df.columns
        if c != "video_id" and pd.api.types.is_numeric_dtype(df[c])
    ]

    for col in columnas_numericas:
        s = pd.to_numeric(df[col], errors="coerce")


        if s.notna().sum() < 5:
            continue


        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1

        if pd.isna(iqr) or iqr == 0:
            limite_inf_iqr = np.nan
            limite_sup_iqr = np.nan
            out_iqr = pd.Series(False, index=df.index)
        else:
            limite_inf_iqr = q1 - 1.5 * iqr
            limite_sup_iqr = q3 + 1.5 * iqr
            out_iqr = (s < limite_inf_iqr) | (s > limite_sup_iqr)


        mediana = s.median()
        mad = np.median(np.abs(s.dropna() - mediana))

        if pd.isna(mad) or mad == 0:
            z_robusto = pd.Series(np.nan, index=df.index)
            out_mad = pd.Series(False, index=df.index)
        else:
            z_robusto = 0.6745 * (s - mediana) / mad
            out_mad = z_robusto.abs() > 3.5


        out_any = out_iqr | out_mad

        for idx in df.index[out_any]:
            resultados.append({
                "dataset": nombre_dataset,
                "video_id": df.loc[idx, "video_id"],
                "variable": col,
                "valor": s.loc[idx],

                "outlier_iqr": bool(out_iqr.loc[idx]),
                "outlier_mad": bool(out_mad.loc[idx]),
                "metodo_outlier": (
                    "IQR+MAD" if out_iqr.loc[idx] and out_mad.loc[idx]
                    else "IQR" if out_iqr.loc[idx]
                    else "MAD"
                ),

                "q1": q1,
                "q3": q3,
                "iqr": iqr,
                "limite_inf_iqr": limite_inf_iqr,
                "limite_sup_iqr": limite_sup_iqr,

                "mediana": mediana,
                "mad": mad,
                "z_robusto": z_robusto.loc[idx],
            })

    return pd.DataFrame(resultados)


outliers_todos = []

for nombre, df_bloque in datasets_bloques.items():
    if df_bloque is None or df_bloque.empty:
        continue

    print(f"Detectando outliers en: {nombre}")

    out = detectar_outliers_iqr_y_mad(df_bloque, nombre)

    if not out.empty:
        outliers_todos.append(out)

outliers_detalle = (
    pd.concat(outliers_todos, ignore_index=True)
    if outliers_todos
    else pd.DataFrame()
)


outliers_detalle.to_csv(
    TABLAS_DIR / "detalle_outliers_por_variable_y_video.csv",
    index=False,
    encoding="utf-8-sig"
)

display(outliers_detalle.head(30))


if not outliers_detalle.empty:
    resumen_outliers_por_video = (
        outliers_detalle
        .groupby(["dataset", "video_id"])
        .agg(
            n_variables_outlier=("variable", "nunique"),
            n_detecciones_outlier=("variable", "count"),
            variables_outlier=("variable", lambda x: ", ".join(sorted(set(x))))
        )
        .reset_index()
        .sort_values(
            ["dataset", "n_variables_outlier", "n_detecciones_outlier"],
            ascending=[True, False, False]
        )
    )
else:
    resumen_outliers_por_video = pd.DataFrame()

resumen_outliers_por_video.to_csv(
    TABLAS_DIR / "resumen_outliers_por_video.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_outliers_por_video)


if not outliers_detalle.empty:
    resumen_outliers_por_variable = (
        outliers_detalle
        .groupby(["dataset", "variable"])
        .agg(
            n_videos_outlier=("video_id", "nunique"),
            videos_outlier=("video_id", lambda x: ", ".join(sorted(set(x)))),
            valor_min_outlier=("valor", "min"),
            valor_max_outlier=("valor", "max"),
        )
        .reset_index()
        .sort_values(
            ["dataset", "n_videos_outlier"],
            ascending=[True, False]
        )
    )
else:
    resumen_outliers_por_variable = pd.DataFrame()

resumen_outliers_por_variable.to_csv(
    TABLAS_DIR / "resumen_outliers_por_variable.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_outliers_por_variable)


if not outliers_detalle.empty:
    resumen_outliers_por_dataset = (
        outliers_detalle
        .groupby("dataset")
        .agg(
            n_videos_con_outliers=("video_id", "nunique"),
            n_variables_con_outliers=("variable", "nunique"),
            n_detecciones_outlier=("variable", "count"),
        )
        .reset_index()
        .sort_values("n_detecciones_outlier", ascending=False)
    )
else:
    resumen_outliers_por_dataset = pd.DataFrame()

resumen_outliers_por_dataset.to_csv(
    TABLAS_DIR / "resumen_outliers_por_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_outliers_por_dataset)


In [ ]:
VIDEOS_REVISAR = ["Longi_MASC_070", "Longi_MASC_052"]

datasets_para_revisar = {
    "eventos_detallado": dataset_eventos_detallado,
    "velocidad_normal_sin_salida": dataset_velocidad_normal_sin_salida,
    "velocidad_salida_frame_comun": dataset_velocidad_salida_frame_comun,
}

revision_presencia = []

for nombre, df in datasets_para_revisar.items():
    for video in VIDEOS_REVISAR:
        esta = video in df["video_id"].values if "video_id" in df.columns else False

        if esta:
            fila = df[df["video_id"] == video].copy()
            n_nulos = int(fila.isna().sum(axis=1).iloc[0])
            n_columnas = fila.shape[1]
        else:
            n_nulos = None
            n_columnas = None

        revision_presencia.append({
            "dataset": nombre,
            "video_id": video,
            "esta_en_dataset": esta,
            "n_columnas_dataset": n_columnas,
            "n_nulos_en_fila": n_nulos,
        })

revision_presencia = pd.DataFrame(revision_presencia)

display(revision_presencia)

revision_presencia.to_csv(
    TABLAS_DIR / "revision_presencia_70_52_por_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
def clasificar_gravedad_outlier(z):
    if pd.isna(z):
        return "sin_z"

    z_abs = abs(z)

    if z_abs >= 8:
        return "critico"
    elif z_abs >= 5:
        return "moderado_alto"
    elif z_abs >= 3.5:
        return "moderado"
    else:
        return "leve"


outliers_detalle_gravedad = outliers_detalle.copy()

outliers_detalle_gravedad["abs_z_robusto"] = outliers_detalle_gravedad["z_robusto"].abs()

outliers_detalle_gravedad["gravedad"] = outliers_detalle_gravedad["z_robusto"].apply(
    clasificar_gravedad_outlier
)


outliers_detalle_gravedad = outliers_detalle_gravedad.sort_values(
    ["abs_z_robusto"],
    ascending=False
)


ruta_outliers_gravedad = TABLAS_DIR / "detalle_outliers_con_gravedad_COMPLETO.csv"

outliers_detalle_gravedad.to_csv(
    ruta_outliers_gravedad,
    index=False,
    encoding="utf-8-sig"
)

print("Tabla completa guardada en:")
print(ruta_outliers_gravedad)

print("Dimensiones de la tabla completa:")
print(outliers_detalle_gravedad.shape)


display(outliers_detalle_gravedad.head(50))


In [ ]:
VIDEOS_EXCLUIR_FINAL = [
    "Longi_MASC_052",
    "Longi_MASC_070",
]

GRAVEDADES_A_NAN = [
    "critico"
]


PATRONES_NO_MODELO = [
    "_px",
    "cadera_x",
    "cadera_y",
    "frame",
    "score",
    "estado",
    "criterio",
    "pose",
    "debug",
    "archivo",
    "fuente",
    "source",
]


def variable_valida_para_nan(variable):
    v = str(variable).lower()
    return not any(p in v for p in PATRONES_NO_MODELO)


def quitar_videos_y_aplicar_nan_criticos(df, nombre_dataset, outliers_gravedad):
    """
    1. Elimina vídeos 52 y 70.
    2. Convierte a NaN los outliers críticos restantes de ese dataset.
    3. Devuelve dataframe limpio y tabla de cambios.
    """

    df_limpio = df.copy()


    df_limpio = asegurar_video_id_primera_columna(df_limpio)


    n_antes = len(df_limpio)

    df_limpio = df_limpio[
        ~df_limpio["video_id"].isin(VIDEOS_EXCLUIR_FINAL)
    ].copy()

    n_despues = len(df_limpio)

    print(f"{nombre_dataset}: vídeos antes = {n_antes}, después = {n_despues}")


    out_criticos = outliers_gravedad[
        (outliers_gravedad["dataset"] == nombre_dataset)
        & (outliers_gravedad["gravedad"].isin(GRAVEDADES_A_NAN))
        & (~outliers_gravedad["video_id"].isin(VIDEOS_EXCLUIR_FINAL))
        & (outliers_gravedad["variable"].apply(variable_valida_para_nan))
    ].copy()

    cambios = []

    for _, row in out_criticos.iterrows():
        video = row["video_id"]
        variable = row["variable"]

        if variable not in df_limpio.columns:
            continue

        mask = df_limpio["video_id"] == video

        if mask.sum() == 0:
            continue

        valor_original = df_limpio.loc[mask, variable].iloc[0]

        if pd.isna(valor_original):
            continue

        df_limpio.loc[mask, variable] = np.nan

        cambios.append({
            "dataset": nombre_dataset,
            "video_id": video,
            "variable": variable,
            "valor_original": valor_original,
            "z_robusto": row.get("z_robusto", np.nan),
            "abs_z_robusto": row.get("abs_z_robusto", np.nan),
            "gravedad": row.get("gravedad", np.nan),
            "metodo_outlier": row.get("metodo_outlier", np.nan),
            "motivo": "outlier_critico_convertido_a_nan"
        })

    df_limpio = asegurar_video_id_primera_columna(df_limpio)

    return df_limpio, pd.DataFrame(cambios)


datasets_bloques_final = {}
cambios_nan_final = []

for nombre, df_bloque in datasets_bloques.items():
    if df_bloque is None or df_bloque.empty:
        continue

    df_limpio, cambios = quitar_videos_y_aplicar_nan_criticos(
        df_bloque,
        nombre,
        outliers_detalle_gravedad
    )

    datasets_bloques_final[nombre] = df_limpio

    if not cambios.empty:
        cambios_nan_final.append(cambios)


cambios_nan_final = (
    pd.concat(cambios_nan_final, ignore_index=True)
    if cambios_nan_final
    else pd.DataFrame()
)

display(cambios_nan_final)

cambios_nan_final.to_csv(
    TABLAS_DIR / "limpieza_final_outliers_criticos_a_nan.csv",
    index=False,
    encoding="utf-8-sig"
)


resumenes_finales = []

for nombre, df_final in datasets_bloques_final.items():

    nombre_final = f"{nombre}_FINAL_sin_52_70_outliers_criticos_nan"


    guardar_csv(
        df_final,
        DATASETS_DIR / f"{nombre_final}_sin_imputar.csv"
    )


    df_imp, resumen_imp = imputar_mediana(df_final)

    guardar_csv(
        df_imp,
        DATASETS_DIR / f"{nombre_final}_imputado_mediana.csv"
    )

    resumen_imp.to_csv(
        TABLAS_DIR / f"imputacion_{nombre_final}.csv",
        index=False,
        encoding="utf-8-sig"
    )


    resumen = resumen_calidad_dataset(df_final, nombre_final)
    resumenes_finales.append(resumen)

    resumen.to_csv(
        TABLAS_DIR / f"calidad_{nombre_final}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(nombre_final, df_final.shape)
    display(df_final.head())


resumen_calidad_final = (
    pd.concat(resumenes_finales, ignore_index=True)
    if resumenes_finales
    else pd.DataFrame()
)

resumen_calidad_final.to_csv(
    TABLAS_DIR / "calidad_final_todos_los_bloques_sin_52_70_outliers_nan.csv",
    index=False,
    encoding="utf-8-sig"
)


def merge_por_video_left(base_df, otros_dfs):
    """
    Une datasets conservando todos los vídeos del dataset base.
    """
    base_df = asegurar_video_id_primera_columna(base_df)
    out = base_df.copy()

    for df_next in otros_dfs:
        if df_next is None or df_next.empty:
            continue

        df_next = asegurar_video_id_primera_columna(df_next)
        out = out.merge(df_next, on="video_id", how="left")

    return asegurar_video_id_primera_columna(out)


dataset_eventos_final = datasets_bloques_final.get("eventos_detallado", pd.DataFrame())
dataset_velocidad_normal_final = datasets_bloques_final.get("velocidad_normal_sin_salida", pd.DataFrame())
dataset_frame_comun_final = datasets_bloques_final.get("velocidad_salida_frame_comun", pd.DataFrame())


combinado_eventos_mas_velocidad_normal_FINAL = merge_por_video_left(
    dataset_eventos_final,
    [dataset_velocidad_normal_final]
)

combinado_eventos_mas_velocidad_normal_mas_frame_comun_FINAL = merge_por_video_left(
    dataset_eventos_final,
    [
        dataset_velocidad_normal_final,
        dataset_frame_comun_final
    ]
)

combinado_velocidad_total_FINAL = merge_por_video_left(
    dataset_velocidad_normal_final,
    [dataset_frame_comun_final]
)


datasets_combinados_final = {
    "combinado_eventos_detallado_mas_velocidad_normal_FINAL": combinado_eventos_mas_velocidad_normal_FINAL,
    "combinado_eventos_detallado_mas_velocidad_normal_mas_frame_comun_FINAL": combinado_eventos_mas_velocidad_normal_mas_frame_comun_FINAL,
    "combinado_velocidad_normal_mas_frame_comun_FINAL": combinado_velocidad_total_FINAL,
}


for nombre, df_comb in datasets_combinados_final.items():

    if df_comb is None or df_comb.empty:
        continue

    df_comb = asegurar_video_id_primera_columna(df_comb)

    guardar_csv(
        df_comb,
        DATASETS_DIR / f"{nombre}_sin_imputar.csv"
    )

    df_comb_imp, resumen_imp_comb = imputar_mediana(df_comb)

    guardar_csv(
        df_comb_imp,
        DATASETS_DIR / f"{nombre}_imputado_mediana.csv"
    )

    resumen_imp_comb.to_csv(
        TABLAS_DIR / f"imputacion_{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    resumen_comb = resumen_calidad_dataset(df_comb, nombre)

    resumen_comb.to_csv(
        TABLAS_DIR / f"calidad_{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(nombre, df_comb.shape)
    display(df_comb.head())


revision_final_videos = []

for nombre, df in {**datasets_bloques_final, **datasets_combinados_final}.items():
    if df is None or df.empty:
        continue

    revision_final_videos.append({
        "dataset": nombre,
        "n_filas": len(df),
        "n_videos": df["video_id"].nunique(),
        "contiene_052": "Longi_MASC_052" in df["video_id"].values,
        "contiene_070": "Longi_MASC_070" in df["video_id"].values,
        "video_id_primera_columna": df.columns[0] == "video_id",
    })

revision_final_videos = pd.DataFrame(revision_final_videos)

display(revision_final_videos)

revision_final_videos.to_csv(
    TABLAS_DIR / "revision_final_videos_sin_52_70.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
datasets_a_comprobar = {
    "eventos_final": dataset_eventos_final,
    "velocidad_normal_final": dataset_velocidad_normal_final,
    "frame_comun_final": dataset_frame_comun_final,
    "combinado_eventos_velocidad_normal": combinado_eventos_mas_velocidad_normal_FINAL,
    "combinado_total": combinado_eventos_mas_velocidad_normal_mas_frame_comun_FINAL,
}

revision_nan_final = []

for nombre, df in datasets_a_comprobar.items():
    revision_nan_final.append({
        "dataset": nombre,
        "n_filas": df.shape[0],
        "n_columnas": df.shape[1],
        "n_nulos_total": df.isna().sum().sum(),
        "n_variables_con_nulos": (df.isna().sum() > 0).sum(),
        "contiene_052": "Longi_MASC_052" in df["video_id"].values,
        "contiene_070": "Longi_MASC_070" in df["video_id"].values,
    })

revision_nan_final = pd.DataFrame(revision_nan_final)

display(revision_nan_final)


In [ ]:
def guardar_correlacion(df, nombre):
    if df is None or df.empty:
        return

    df = asegurar_video_id_primera_columna(df)
    num_cols = [
        c for c in df.columns
        if c != "video_id" and pd.api.types.is_numeric_dtype(df[c])
    ]

    if len(num_cols) < 2:
        print(f"{nombre}: no hay suficientes variables numéricas para correlación.")
        return

    corr = df[num_cols].corr()

    corr.reset_index().rename(columns={"index": "variable"}).to_csv(
        TABLAS_DIR / f"correlacion_{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    plt.figure(figsize=(max(7, 0.30 * len(num_cols)), max(6, 0.30 * len(num_cols))))
    plt.imshow(corr, aspect="auto")
    plt.colorbar(label="Correlación")
    plt.xticks(range(len(num_cols)), num_cols, rotation=90)
    plt.yticks(range(len(num_cols)), num_cols)
    plt.title(f"Matriz de correlación - {nombre}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"correlacion_{nombre}.png", dpi=300, bbox_inches="tight")
    plt.show()


datasets_para_correlacion = {}

if "datasets_bloques" in globals():
    datasets_para_correlacion.update({
        f"{nombre}_inicial": df
        for nombre, df in datasets_bloques.items()
    })

if "datasets_bloques_final" in globals():
    datasets_para_correlacion.update({
        f"{nombre}_final": df
        for nombre, df in datasets_bloques_final.items()
    })

if "datasets_combinados_final" in globals():
    datasets_para_correlacion.update(datasets_combinados_final)

for nombre, df_bloque in datasets_para_correlacion.items():
    guardar_correlacion(df_bloque, nombre)


In [ ]:
todos_datasets_revision = {}

if "datasets_bloques" in globals():
    todos_datasets_revision.update({
        f"{nombre}_inicial": df
        for nombre, df in datasets_bloques.items()
    })

if "datasets_bloques_final" in globals():
    todos_datasets_revision.update({
        f"{nombre}_final": df
        for nombre, df in datasets_bloques_final.items()
    })

if "datasets_combinados_final" in globals():
    todos_datasets_revision.update(datasets_combinados_final)

revision_video_id = []

for nombre, df in todos_datasets_revision.items():
    if df is None or df.empty:
        continue

    revision_video_id.append({
        "dataset": nombre,
        "tiene_video_id": "video_id" in df.columns,
        "primera_columna": df.columns[0] if len(df.columns) else None,
        "n_videos": df["video_id"].nunique() if "video_id" in df.columns else np.nan,
        "n_filas": len(df),
    })

revision_video_id = pd.DataFrame(revision_video_id)

display(revision_video_id)

revision_video_id.to_csv(
    TABLAS_DIR / "02_revision_video_id.csv",
    index=False,
    encoding="utf-8-sig"
)


## 5. Merge competition results and athlete reference level

Biomechanical variables are joined with official attempt results and a historical athlete-level reference. The resulting table is the single input used by the modelling sections below.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"

TABLAS_DIR = SALIDA_DIR / "tablas"
DATASETS_DIR = SALIDA_DIR / "datasets"
FIGURAS_DIR = SALIDA_DIR / "figuras"

TABLAS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta principal:", SALIDA_DIR.resolve())
print("Tablas:", TABLAS_DIR.resolve())
print("Datasets:", DATASETS_DIR.resolve())
print("Figuras:", FIGURAS_DIR.resolve())


todos_datasets = {}

if "datasets_bloques" in globals():
    todos_datasets.update(datasets_bloques)
else:
    print("Aviso: no existe datasets_bloques.")

if "datasets_combinados" in globals():
    todos_datasets.update(datasets_combinados)
else:
    print("Aviso: no existe datasets_combinados. Se guardan solo datasets_bloques.")


posibles_datasets_sueltos = {
    "eventos_final": globals().get("dataset_eventos_final"),
    "velocidad_normal_final": globals().get("dataset_velocidad_normal_final"),
    "frame_comun_final": globals().get("dataset_frame_comun_final"),
    "combinado_eventos_velocidad_normal_FINAL": globals().get("combinado_eventos_mas_velocidad_normal_FINAL"),
    "combinado_total_FINAL": globals().get("combinado_eventos_mas_velocidad_normal_mas_frame_comun_FINAL"),
}

for nombre, df in posibles_datasets_sueltos.items():
    if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
        todos_datasets[nombre] = df


rutas_datasets_guardados = []

for nombre, df in todos_datasets.items():
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        continue

    df_guardar = df.copy()

    if "video_id" in df_guardar.columns:
        columnas = ["video_id"] + [c for c in df_guardar.columns if c != "video_id"]
        df_guardar = df_guardar[columnas]

    ruta_salida = DATASETS_DIR / f"{nombre}.csv"

    df_guardar.to_csv(
        ruta_salida,
        index=False,
        encoding="utf-8-sig"
    )

    rutas_datasets_guardados.append({
        "dataset": nombre,
        "ruta": str(ruta_salida),
        "n_filas": df_guardar.shape[0],
        "n_columnas": df_guardar.shape[1],
        "n_videos": df_guardar["video_id"].nunique() if "video_id" in df_guardar.columns else np.nan,
    })

rutas_datasets_guardados = pd.DataFrame(rutas_datasets_guardados)

display(rutas_datasets_guardados)

rutas_datasets_guardados.to_csv(
    TABLAS_DIR / "03_rutas_datasets_guardados.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata


BASE_DIR = Path(".")

DATOS_DIR = BASE_DIR / "datos"

SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
TABLAS_DIR = SALIDA_DIR / "tablas"
DATASETS_DIR = SALIDA_DIR / "datasets"
FIGURAS_DIR = SALIDA_DIR / "figuras"

TABLAS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)

RUTA_RESULTADOS = DATOS_DIR / "DEFINITIVO.xlsx"

print("Carpeta datos:", DATOS_DIR.resolve())
print("Carpeta datasets:", DATASETS_DIR.resolve())
print("Archivo de resultados:", RUTA_RESULTADOS.resolve())


def normalizar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto)

    return texto


def crear_id_atleta(nombre):
    nombre = normalizar_texto(nombre)
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    nombre = nombre.strip("_")

    return nombre


def normalizar_video(video):
    if pd.isna(video):
        return ""

    texto = str(video).strip()
    texto = Path(texto).stem

    m = re.search(r"Longi[_\s-]*MASC[_\s-]*(\d+)", texto, flags=re.IGNORECASE)

    if m:
        num = int(m.group(1))
        return f"Longi_MASC_{num:03d}"

    return texto


def limpiar_marca(valor):
    """
    Convierte el resultado del salto a metros.
    Si es nulo, X, NM, etc., devuelve NaN.
    """
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().lower().replace(",", ".")

    valores_nulos = [
        "", "x", "xx", "xxx", "nm", "dns", "dnf", "dq",
        "np", "-", "--", "nan", "nulo"
    ]

    if texto in valores_nulos:
        return np.nan

    m = re.search(r"(\d+(?:\.\d+)?)", texto)

    if not m:
        return np.nan

    return float(m.group(1))


def detectar_nulo(valor):
    """
    Devuelve:
    - 1 si el resultado indica salto nulo.
    - 0 si hay marca válida.
    - NaN si no se puede interpretar.
    """
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().lower().replace(",", ".")

    valores_nulos = ["x", "xx", "xxx", "nm", "dq", "nulo"]

    if texto in valores_nulos:
        return 1

    marca = limpiar_marca(valor)

    if pd.notna(marca):
        return 0

    return np.nan


def encontrar_columna(df, posibles):
    """
    Busca una columna aunque el nombre no coincida exactamente.
    """
    columnas_norm = {normalizar_texto(c): c for c in df.columns}

    for posible in posibles:
        posible_norm = normalizar_texto(posible)

        for col_norm, col_real in columnas_norm.items():
            if posible_norm == col_norm or posible_norm in col_norm:
                return col_real

    return None


def asegurar_video_id_primera_columna(df):
    df = df.copy()

    if "video_id" in df.columns:
        columnas = ["video_id"] + [c for c in df.columns if c != "video_id"]
        df = df[columnas]

    return df


if not RUTA_RESULTADOS.exists():
    raise FileNotFoundError(
        f"No encuentro DEFINITIVO.xlsx en:\n{RUTA_RESULTADOS.resolve()}\n\n"
        "Mete el archivo dentro de la carpeta datos."
    )

resultados_raw = pd.read_excel(RUTA_RESULTADOS)

print("Columnas del archivo DEFINITIVO.xlsx:")
print(resultados_raw.columns.tolist())

display(resultados_raw.head())


col_video = encontrar_columna(
    resultados_raw,
    ["video_id", "video", "archivo_video", "nombre_video"]
)

col_atleta = encontrar_columna(
    resultados_raw,
    ["atleta", "nombre_atleta", "nombre", "Columna1"]
)

col_resultado = encontrar_columna(
    resultados_raw,
    ["resultado", "marca", "marca_m", "result"]
)

if col_video is None:
    raise ValueError("No encuentro la columna del vídeo en DEFINITIVO.xlsx.")

if col_atleta is None:
    raise ValueError("No encuentro la columna del atleta en DEFINITIVO.xlsx.")

if col_resultado is None:
    raise ValueError("No encuentro la columna del resultado/marca en DEFINITIVO.xlsx.")

print("Columna vídeo detectada:", col_video)
print("Columna atleta detectada:", col_atleta)
print("Columna resultado detectada:", col_resultado)


resultados_limpios = resultados_raw.copy()

resultados_limpios["video_id"] = resultados_limpios[col_video].apply(normalizar_video)
resultados_limpios["atleta"] = resultados_limpios[col_atleta].astype(str).str.strip()
resultados_limpios["id_atleta"] = resultados_limpios["atleta"].apply(crear_id_atleta)

resultados_limpios["resultado_original"] = resultados_limpios[col_resultado]
resultados_limpios["marca_m"] = resultados_limpios["resultado_original"].apply(limpiar_marca)
resultados_limpios["resultado_nulo"] = resultados_limpios["resultado_original"].apply(detectar_nulo)


columnas_resultados = [
    "video_id",
    "id_salto",
    "competicion",
    "intento",
    "atleta",
    "id_atleta",
    "resultado_original",
    "marca_m",
    "resultado_nulo",
]

columnas_resultados = [
    c for c in columnas_resultados
    if c in resultados_limpios.columns
]

resultados_limpios = resultados_limpios[columnas_resultados].copy()
resultados_limpios = asegurar_video_id_primera_columna(resultados_limpios)


duplicados = resultados_limpios[
    resultados_limpios["video_id"].duplicated(keep=False)
].copy()

if not duplicados.empty:
    print("AVISO: hay vídeos duplicados en la tabla de resultados.")
    display(duplicados.sort_values("video_id"))


ruta_resultados_limpios = TABLAS_DIR / "resultados_saltos_limpios_por_video.csv"

resultados_limpios.to_csv(
    ruta_resultados_limpios,
    index=False,
    encoding="utf-8-sig"
)

print("Tabla limpia de resultados guardada en:")
print(ruta_resultados_limpios)

display(resultados_limpios.head())
print("Dimensiones resultados_limpios:", resultados_limpios.shape)


def cargar_dataset_final(nombre_archivo):
    ruta = DATASETS_DIR / nombre_archivo

    if not ruta.exists():
        raise FileNotFoundError(f"No encuentro el dataset:\n{ruta.resolve()}")

    df = pd.read_csv(ruta, encoding="utf-8-sig")
    df["video_id"] = df["video_id"].apply(normalizar_video)
    df = asegurar_video_id_primera_columna(df)

    return df


eventos_final = cargar_dataset_final("eventos_final.csv")
velocidad_normal_final = cargar_dataset_final("velocidad_normal_final.csv")
frame_comun_final = cargar_dataset_final("frame_comun_final.csv")
combinado_eventos_velocidad_normal_FINAL = cargar_dataset_final("combinado_eventos_velocidad_normal_FINAL.csv")
combinado_total_FINAL = cargar_dataset_final("combinado_total_FINAL.csv")

datasets_finales = {
    "eventos_final": eventos_final,
    "velocidad_normal_final": velocidad_normal_final,
    "frame_comun_final": frame_comun_final,
    "combinado_eventos_velocidad_normal_FINAL": combinado_eventos_velocidad_normal_FINAL,
    "combinado_total_FINAL": combinado_total_FINAL,
}


def anadir_resultados_a_dataset(df, nombre_dataset, resultados):
    df = df.copy()

    if "video_id" not in df.columns:
        raise ValueError(f"{nombre_dataset} no tiene columna video_id.")

    df["video_id"] = df["video_id"].apply(normalizar_video)


    columnas_resultado = [
        "id_salto",
        "competicion",
        "intento",
        "atleta",
        "id_atleta",
        "resultado_original",
        "marca_m",
        "resultado_nulo",
    ]

    columnas_a_eliminar = [
        c for c in columnas_resultado
        if c in df.columns
    ]

    if columnas_a_eliminar:
        df = df.drop(columns=columnas_a_eliminar)

    df_con_resultado = df.merge(
        resultados,
        on="video_id",
        how="left"
    )

    df_con_resultado = asegurar_video_id_primera_columna(df_con_resultado)

    return df_con_resultado


datasets_finales_con_resultado = {}
revision_union_resultados = []

for nombre, df in datasets_finales.items():
    df_con_resultado = anadir_resultados_a_dataset(
        df=df,
        nombre_dataset=nombre,
        resultados=resultados_limpios
    )

    datasets_finales_con_resultado[nombre] = df_con_resultado

    ruta_salida = DATASETS_DIR / f"{nombre}_con_resultado.csv"

    df_con_resultado.to_csv(
        ruta_salida,
        index=False,
        encoding="utf-8-sig"
    )

    revision_union_resultados.append({
        "dataset": nombre,
        "n_filas": len(df_con_resultado),
        "n_columnas": df_con_resultado.shape[1],
        "n_videos": df_con_resultado["video_id"].nunique(),
        "sin_atleta": df_con_resultado["atleta"].isna().sum(),
        "sin_resultado_original": df_con_resultado["resultado_original"].isna().sum(),
        "sin_resultado_nulo": df_con_resultado["resultado_nulo"].isna().sum(),
        "n_saltos_validos": int((df_con_resultado["resultado_nulo"] == 0).sum()),
        "n_saltos_nulos": int((df_con_resultado["resultado_nulo"] == 1).sum()),
        "n_marcas_validas": int(df_con_resultado["marca_m"].notna().sum()),
        "ruta_guardado": str(ruta_salida),
    })

    print(f"Guardado: {ruta_salida}")


revision_union_resultados = pd.DataFrame(revision_union_resultados)

display(revision_union_resultados)

revision_union_resultados.to_csv(
    TABLAS_DIR / "revision_union_resultados_datasets_finales.csv",
    index=False,
    encoding="utf-8-sig"
)


eventos_final_con_resultado = datasets_finales_con_resultado["eventos_final"]
velocidad_normal_final_con_resultado = datasets_finales_con_resultado["velocidad_normal_final"]
frame_comun_final_con_resultado = datasets_finales_con_resultado["frame_comun_final"]
combinado_eventos_velocidad_normal_FINAL_con_resultado = datasets_finales_con_resultado["combinado_eventos_velocidad_normal_FINAL"]
combinado_total_FINAL_con_resultado = datasets_finales_con_resultado["combinado_total_FINAL"]

print("Dataset principal para seguir:")
print("combinado_total_FINAL_con_resultado")
display(combinado_total_FINAL_con_resultado.head())


ruta_para_nivel = DATOS_DIR / "dataset_ml_saltos_con_resultado.csv"

combinado_total_FINAL_con_resultado.to_csv(
    ruta_para_nivel,
    index=False,
    encoding="utf-8-sig"
)

print("Copia preparada para calcular nivel de atleta:")
print(ruta_para_nivel.resolve())


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata


BASE_DIR = Path(".")

ATLETAS_DIR = BASE_DIR / "atletas"
DATOS_DIR = BASE_DIR / "datos"

SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
TABLAS_DIR = SALIDA_DIR / "tablas"
DATASETS_DIR = SALIDA_DIR / "datasets"
NIVEL_DIR = SALIDA_DIR / "nivel_atleta"

TABLAS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
NIVEL_DIR.mkdir(parents=True, exist_ok=True)


FECHA_COMPETICION = pd.to_datetime("2025-02-21")


INICIO_PC_ACTUAL = pd.to_datetime("2024-12-01")

print("Carpeta atletas:", ATLETAS_DIR.resolve())
print("Carpeta datasets:", DATASETS_DIR.resolve())
print("Carpeta nivel atleta:", NIVEL_DIR.resolve())


def normalizar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto)
    return texto


def crear_id_atleta(nombre):
    nombre = normalizar_texto(nombre)
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    nombre = nombre.strip("_")
    return nombre


def limpiar_marca(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().lower().replace(",", ".")

    valores_no_validos = [
        "", "x", "xx", "xxx", "nm", "dns", "dnf", "dq",
        "np", "-", "--", "nan", "nulo"
    ]

    if texto in valores_no_validos:
        return np.nan

    m = re.search(r"(\d+(?:\.\d+)?)", texto)

    if not m:
        return np.nan

    return float(m.group(1))


def leer_csv_auto(ruta):
    ruta = Path(ruta)

    for sep in [",", ";", "\t"]:
        try:
            df = pd.read_csv(ruta, sep=sep, encoding="utf-8-sig")
            if df.shape[1] > 1:
                return df
        except Exception:
            pass

    return pd.read_csv(ruta, encoding="utf-8-sig")


def encontrar_columna(df, posibles):
    columnas_norm = {
        normalizar_texto(c): c
        for c in df.columns
    }

    for posible in posibles:
        posible_norm = normalizar_texto(posible)

        for col_norm, col_real in columnas_norm.items():
            if posible_norm == col_norm or posible_norm in col_norm:
                return col_real

    return None


def media_top_n(serie, n=3):
    serie = pd.to_numeric(serie, errors="coerce")
    serie = serie.dropna().sort_values(ascending=False)

    if len(serie) == 0:
        return np.nan

    return serie.head(n).mean()


def asegurar_video_id_primera_columna(df):
    df = df.copy()

    if "video_id" in df.columns:
        columnas = ["video_id"] + [c for c in df.columns if c != "video_id"]
        df = df[columnas]

    return df


if "combinado_total_FINAL_con_resultado" in globals():
    df_modelo_base = combinado_total_FINAL_con_resultado.copy()
    print("Usando combinado_total_FINAL_con_resultado desde memoria.")
else:
    ruta_dataset_resultado = DATASETS_DIR / "combinado_total_FINAL_con_resultado.csv"

    if not ruta_dataset_resultado.exists():
        raise FileNotFoundError(
            f"No encuentro el dataset con resultado en:\n{ruta_dataset_resultado.resolve()}\n\n"
            "Primero tienes que ejecutar el bloque que añade atleta, marca y resultado."
        )

    df_modelo_base = pd.read_csv(ruta_dataset_resultado, encoding="utf-8-sig")
    print("Dataset con resultado cargado desde:")
    print(ruta_dataset_resultado.resolve())


columnas_necesarias = ["video_id", "atleta", "marca_m", "resultado_nulo"]

for col in columnas_necesarias:
    if col not in df_modelo_base.columns:
        raise ValueError(f"Falta la columna necesaria: {col}")


if "id_atleta" not in df_modelo_base.columns:
    df_modelo_base["id_atleta"] = df_modelo_base["atleta"].apply(crear_id_atleta)

df_modelo_base["id_atleta"] = df_modelo_base["id_atleta"].astype(str)
df_modelo_base["marca_m"] = df_modelo_base["marca_m"].apply(limpiar_marca)

print("Dataset base con resultado:")
print(df_modelo_base.shape)

display(
    df_modelo_base[
        ["video_id", "atleta", "id_atleta", "marca_m", "resultado_nulo"]
    ].head(10)
)


archivos_historicos = sorted(ATLETAS_DIR.glob("*_limpio_pc_longitud.csv"))

if len(archivos_historicos) == 0:
    raise FileNotFoundError(
        f"No he encontrado archivos *_limpio_pc_longitud.csv en:\n{ATLETAS_DIR.resolve()}"
    )

historicos = []

for archivo in archivos_historicos:
    df_hist = leer_csv_auto(archivo)

    col_fecha = encontrar_columna(df_hist, ["fecha", "date"])
    col_atleta = encontrar_columna(df_hist, ["atleta", "nombre", "athlete"])
    col_marca = encontrar_columna(df_hist, ["marca", "resultado", "mark"])

    if col_fecha is None or col_atleta is None or col_marca is None:
        print(f"AVISO: se salta {archivo.name} porque no encuentro fecha/atleta/marca.")
        continue

    tmp = df_hist.copy()

    tmp["archivo_origen"] = archivo.name
    tmp["fecha"] = pd.to_datetime(tmp[col_fecha], errors="coerce")
    tmp["atleta"] = tmp[col_atleta].astype(str).str.strip()
    tmp["id_atleta"] = tmp["atleta"].apply(crear_id_atleta)
    tmp["marca_m"] = tmp[col_marca].apply(limpiar_marca)

    tmp = tmp.dropna(subset=["fecha", "marca_m"])


    tmp = tmp[tmp["fecha"] < FECHA_COMPETICION].copy()

    historicos.append(
        tmp[
            [
                "id_atleta",
                "atleta",
                "fecha",
                "marca_m",
                "archivo_origen",
            ]
        ]
    )

if len(historicos) == 0:
    raise ValueError("No se ha podido cargar ningún histórico válido.")

historico_pc_previo = pd.concat(historicos, ignore_index=True)

historico_pc_previo.to_csv(
    NIVEL_DIR / "historico_pc_longitud_previo_usado_para_nivel.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Histórico previo cargado:")
print(historico_pc_previo.shape)
print("Atletas con histórico:", historico_pc_previo["id_atleta"].nunique())


filas_nivel = []

for id_atleta, df_atleta in historico_pc_previo.groupby("id_atleta"):

    df_atleta = df_atleta.sort_values("fecha").copy()

    atleta = df_atleta["atleta"].dropna().iloc[0]


    df_actual = df_atleta[
        (df_atleta["fecha"] >= INICIO_PC_ACTUAL)
        & (df_atleta["fecha"] < FECHA_COMPETICION)
    ].copy()


    df_total = df_atleta[
        df_atleta["fecha"] < FECHA_COMPETICION
    ].copy()

    n_actual = len(df_actual)
    n_total = len(df_total)


    if n_actual >= 3:
        nivel_referencia_pc = media_top_n(df_actual["marca_m"], n=3)
        metodo_nivel = "media_top3_pc_actual_previa"
        fiabilidad_nivel = "alta"
        n_resultados_usados = 3

    elif n_actual == 2:
        nivel_referencia_pc = df_actual["marca_m"].mean()
        metodo_nivel = "media_2_resultados_pc_actual_previa"
        fiabilidad_nivel = "media"
        n_resultados_usados = 2

    elif n_actual == 1:
        nivel_referencia_pc = df_actual["marca_m"].iloc[0]
        metodo_nivel = "unico_resultado_pc_actual_previo"
        fiabilidad_nivel = "baja"
        n_resultados_usados = 1

    elif n_total >= 3:
        nivel_referencia_pc = media_top_n(df_total["marca_m"], n=3)
        metodo_nivel = "media_top3_pc_historica_previa"
        fiabilidad_nivel = "media_baja"
        n_resultados_usados = 3

    elif n_total == 2:
        nivel_referencia_pc = df_total["marca_m"].mean()
        metodo_nivel = "media_2_resultados_pc_historica_previa"
        fiabilidad_nivel = "baja"
        n_resultados_usados = 2

    elif n_total == 1:
        nivel_referencia_pc = df_total["marca_m"].iloc[0]
        metodo_nivel = "unico_resultado_pc_historico_previo"
        fiabilidad_nivel = "muy_baja"
        n_resultados_usados = 1

    else:
        nivel_referencia_pc = np.nan
        metodo_nivel = "nivel_no_calculable"
        fiabilidad_nivel = "no_calculable"
        n_resultados_usados = 0

    filas_nivel.append({
        "id_atleta": id_atleta,
        "atleta": atleta,
        "nivel_referencia_pc": nivel_referencia_pc,
        "metodo_nivel": metodo_nivel,
        "fiabilidad_nivel": fiabilidad_nivel,
        "n_resultados_usados_nivel": n_resultados_usados,
        "n_resultados_pc_actual_previos": n_actual,
        "n_resultados_pc_total_previos": n_total,
        "mejor_marca_previa": df_total["marca_m"].max() if n_total > 0 else np.nan,
        "media_top3_historica_previa": media_top_n(df_total["marca_m"], n=3),
    })

niveles_atleta = pd.DataFrame(filas_nivel)

niveles_atleta = niveles_atleta.sort_values(
    "nivel_referencia_pc",
    ascending=False
).reset_index(drop=True)

niveles_atleta.to_csv(
    NIVEL_DIR / "atletas_nivel_referencia_TFG.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Nivel de referencia por atleta:")
display(niveles_atleta)

print("Tabla completa de niveles guardada en:")
print((NIVEL_DIR / "atletas_nivel_referencia_TFG.csv").resolve())


df_modelo_con_nivel = df_modelo_base.copy()


columnas_repetidas = [
    "nivel_referencia_pc",
    "marca_vs_nivel_cm",
]

columnas_repetidas = [
    c for c in columnas_repetidas
    if c in df_modelo_con_nivel.columns
]

if columnas_repetidas:
    df_modelo_con_nivel = df_modelo_con_nivel.drop(columns=columnas_repetidas)


df_modelo_con_nivel = df_modelo_con_nivel.merge(
    niveles_atleta[["id_atleta", "nivel_referencia_pc"]],
    on="id_atleta",
    how="left"
)


df_modelo_con_nivel["marca_vs_nivel_cm"] = (
    (df_modelo_con_nivel["marca_m"] - df_modelo_con_nivel["nivel_referencia_pc"]) * 100
)

df_modelo_con_nivel = asegurar_video_id_primera_columna(df_modelo_con_nivel)


ruta_dataset_con_nivel = DATASETS_DIR / "combinado_total_FINAL_con_resultado_y_nivel.csv"

df_modelo_con_nivel.to_csv(
    ruta_dataset_con_nivel,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset final con nivel guardado en:")
print(ruta_dataset_con_nivel.resolve())

print("Dimensiones:")
print(df_modelo_con_nivel.shape)

display(
    df_modelo_con_nivel[
        [
            "video_id",
            "atleta",
            "marca_m",
            "resultado_nulo",
            "nivel_referencia_pc",
            "marca_vs_nivel_cm",
        ]
    ].head(20)
)


revision_dataset_nivel = pd.DataFrame([{
    "dataset": "combinado_total_FINAL_con_resultado_y_nivel",
    "n_filas": len(df_modelo_con_nivel),
    "n_columnas": df_modelo_con_nivel.shape[1],
    "n_videos": df_modelo_con_nivel["video_id"].nunique(),
    "sin_nivel_referencia": int(df_modelo_con_nivel["nivel_referencia_pc"].isna().sum()),
    "sin_marca_m": int(df_modelo_con_nivel["marca_m"].isna().sum()),
    "sin_marca_vs_nivel_cm": int(df_modelo_con_nivel["marca_vs_nivel_cm"].isna().sum()),
    "n_saltos_validos": int((df_modelo_con_nivel["resultado_nulo"] == 0).sum()),
    "n_saltos_nulos": int((df_modelo_con_nivel["resultado_nulo"] == 1).sum()),
}])

display(revision_dataset_nivel)

revision_dataset_nivel.to_csv(
    NIVEL_DIR / "revision_dataset_con_nivel_TFG.csv",
    index=False,
    encoding="utf-8-sig"
)


combinado_total_FINAL_con_resultado_y_nivel = df_modelo_con_nivel.copy()


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
TABLAS_DIR = SALIDA_DIR / "tablas"
DATASETS_DIR = SALIDA_DIR / "datasets"
FIGURAS_DIR = SALIDA_DIR / "figuras"
NIVEL_DIR = SALIDA_DIR / "nivel_atleta"

TABLAS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)
NIVEL_DIR.mkdir(parents=True, exist_ok=True)

if "combinado_total_FINAL_con_resultado_y_nivel" in globals():
    df_ml = combinado_total_FINAL_con_resultado_y_nivel.copy()
    print("Usando combinado_total_FINAL_con_resultado_y_nivel desde memoria.")
else:
    ruta_df_ml = DATASETS_DIR / "combinado_total_FINAL_con_resultado_y_nivel.csv"

    if not ruta_df_ml.exists():
        raise FileNotFoundError(
            f"No encuentro el dataset final para ML en:\n{ruta_df_ml.resolve()}"
        )

    df_ml = pd.read_csv(ruta_df_ml, encoding="utf-8-sig")
    print("Dataset final para ML cargado desde:")
    print(ruta_df_ml.resolve())

print("Dimensiones dataset final ML:", df_ml.shape)


columnas_no_biomecanicas = [
    "video_id",
    "id_salto",
    "competicion",
    "intento",
    "atleta",
    "id_atleta",
    "resultado_original",
    "marca_m",
    "resultado_nulo",
    "nivel_referencia_pc",
    "marca_vs_nivel_cm",
]

columnas_no_biomecanicas = [
    c for c in columnas_no_biomecanicas
    if c in df_ml.columns
]

columnas_biomecanicas = [
    c for c in df_ml.columns
    if c not in columnas_no_biomecanicas
]


n_total_valores = df_ml.shape[0] * df_ml.shape[1]
n_nulos_total = int(df_ml.isna().sum().sum())
pct_nulos_total = (n_nulos_total / n_total_valores) * 100 if n_total_valores > 0 else 0

n_total_biomecanicos = df_ml[columnas_biomecanicas].shape[0] * df_ml[columnas_biomecanicas].shape[1]
n_nulos_biomecanicos = int(df_ml[columnas_biomecanicas].isna().sum().sum())
pct_nulos_biomecanicos = (
    (n_nulos_biomecanicos / n_total_biomecanicos) * 100
    if n_total_biomecanicos > 0
    else 0
)

resumen_nulos_ml = pd.DataFrame([{
    "dataset": "combinado_total_FINAL_con_resultado_y_nivel",
    "n_filas": df_ml.shape[0],
    "n_columnas": df_ml.shape[1],
    "n_nulos_total": n_nulos_total,
    "porcentaje_nulos_total": pct_nulos_total,
    "n_columnas_biomecanicas": len(columnas_biomecanicas),
    "n_nulos_biomecanicos": n_nulos_biomecanicos,
    "porcentaje_nulos_biomecanicos": pct_nulos_biomecanicos,
    "n_nulos_marca_m": int(df_ml["marca_m"].isna().sum()) if "marca_m" in df_ml.columns else np.nan,
    "n_nulos_resultado_nulo": int(df_ml["resultado_nulo"].isna().sum()) if "resultado_nulo" in df_ml.columns else np.nan,
    "n_nulos_nivel_referencia_pc": int(df_ml["nivel_referencia_pc"].isna().sum()) if "nivel_referencia_pc" in df_ml.columns else np.nan,
    "n_nulos_marca_vs_nivel_cm": int(df_ml["marca_vs_nivel_cm"].isna().sum()) if "marca_vs_nivel_cm" in df_ml.columns else np.nan,
}])

display(resumen_nulos_ml)

resumen_nulos_ml.to_csv(
    TABLAS_DIR / "revision_nulos_dataset_final_ML_resumen.csv",
    index=False,
    encoding="utf-8-sig"
)


nulos_por_variable_ml = (
    df_ml
    .isna()
    .sum()
    .reset_index()
)

nulos_por_variable_ml.columns = ["variable", "n_nulos"]

nulos_por_variable_ml = nulos_por_variable_ml[
    nulos_por_variable_ml["n_nulos"] > 0
].copy()

nulos_por_variable_ml["porcentaje_nulos"] = (
    nulos_por_variable_ml["n_nulos"] / len(df_ml) * 100
)

def clasificar_tipo_variable_nulos(variable):
    if variable in ["marca_m", "resultado_original", "resultado_nulo"]:
        return "resultado_salto"
    elif variable in ["nivel_referencia_pc", "marca_vs_nivel_cm"]:
        return "nivel_rendimiento"
    elif variable in ["video_id", "id_salto", "competicion", "intento", "atleta", "id_atleta"]:
        return "identificacion"
    else:
        return "biomecanica"

def interpretar_motivo_nulo(variable):
    if variable == "marca_m":
        return "salto_nulo_sin_marca_valida"
    elif variable == "marca_vs_nivel_cm":
        return "salto_nulo_o_falta_marca/nivel"
    elif variable == "nivel_referencia_pc":
        return "atleta_sin_historico_suficiente"
    elif variable == "resultado_nulo":
        return "resultado_no_interpretable"
    else:
        return "valor_biomecanico_faltante_o_outlier_critico_a_nan"

nulos_por_variable_ml["tipo_variable"] = nulos_por_variable_ml["variable"].apply(
    clasificar_tipo_variable_nulos
)

nulos_por_variable_ml["interpretacion"] = nulos_por_variable_ml["variable"].apply(
    interpretar_motivo_nulo
)

nulos_por_variable_ml = nulos_por_variable_ml.sort_values(
    ["tipo_variable", "n_nulos"],
    ascending=[True, False]
)

display(nulos_por_variable_ml)

nulos_por_variable_ml.to_csv(
    TABLAS_DIR / "revision_nulos_dataset_final_ML_por_variable.csv",
    index=False,
    encoding="utf-8-sig"
)


nulos_por_video_ml = pd.DataFrame({
    "video_id": df_ml["video_id"],
    "n_nulos_total": df_ml.isna().sum(axis=1),
    "n_nulos_biomecanicos": df_ml[columnas_biomecanicas].isna().sum(axis=1),
})


for col in ["atleta", "resultado_original", "marca_m", "resultado_nulo", "nivel_referencia_pc", "marca_vs_nivel_cm"]:
    if col in df_ml.columns:
        nulos_por_video_ml[col] = df_ml[col]


def variables_con_nan_fila(row):
    vars_nan = row.index[row.isna()].tolist()
    return ", ".join(vars_nan)

nulos_por_video_ml["variables_con_nan"] = df_ml.apply(
    variables_con_nan_fila,
    axis=1
)

nulos_por_video_ml = nulos_por_video_ml[
    nulos_por_video_ml["n_nulos_total"] > 0
].copy()

nulos_por_video_ml = nulos_por_video_ml.sort_values(
    "n_nulos_total",
    ascending=False
)

display(nulos_por_video_ml)

nulos_por_video_ml.to_csv(
    TABLAS_DIR / "revision_nulos_dataset_final_ML_por_video.csv",
    index=False,
    encoding="utf-8-sig"
)


comprobaciones_nulos_ml = []


if "marca_m" in df_ml.columns and "resultado_nulo" in df_ml.columns:
    filas_marca_nan = df_ml[df_ml["marca_m"].isna()].copy()

    comprobaciones_nulos_ml.append({
        "comprobacion": "marca_m_nan_son_saltos_nulos",
        "n_casos": len(filas_marca_nan),
        "n_casos_resultado_nulo_1": int((filas_marca_nan["resultado_nulo"] == 1).sum()),
        "correcto": len(filas_marca_nan) == int((filas_marca_nan["resultado_nulo"] == 1).sum())
    })


if "marca_vs_nivel_cm" in df_ml.columns:
    filas_target_nan = df_ml[df_ml["marca_vs_nivel_cm"].isna()].copy()

    causa_target_nan = (
        filas_target_nan["marca_m"].isna()
        | filas_target_nan["nivel_referencia_pc"].isna()
        if "marca_m" in df_ml.columns and "nivel_referencia_pc" in df_ml.columns
        else pd.Series([False] * len(filas_target_nan))
    )

    comprobaciones_nulos_ml.append({
        "comprobacion": "marca_vs_nivel_cm_nan_tiene_causa_marca_o_nivel",
        "n_casos": len(filas_target_nan),
        "n_casos_con_causa_identificada": int(causa_target_nan.sum()),
        "correcto": len(filas_target_nan) == int(causa_target_nan.sum())
    })


if "nivel_referencia_pc" in df_ml.columns:
    filas_sin_nivel = df_ml[df_ml["nivel_referencia_pc"].isna()].copy()

    comprobaciones_nulos_ml.append({
        "comprobacion": "atletas_sin_nivel_referencia",
        "n_casos": len(filas_sin_nivel),
        "n_atletas_afectados": filas_sin_nivel["id_atleta"].nunique() if "id_atleta" in filas_sin_nivel.columns else np.nan,
        "correcto": len(filas_sin_nivel) == 0
    })

comprobaciones_nulos_ml = pd.DataFrame(comprobaciones_nulos_ml)

display(comprobaciones_nulos_ml)

comprobaciones_nulos_ml.to_csv(
    TABLAS_DIR / "revision_nulos_dataset_final_ML_comprobaciones.csv",
    index=False,
    encoding="utf-8-sig"
)


if not nulos_por_variable_ml.empty:
    import matplotlib.pyplot as plt

    fig_df = nulos_por_variable_ml.sort_values("n_nulos", ascending=True)

    plt.figure(figsize=(10, max(4, 0.35 * len(fig_df))))
    plt.barh(fig_df["variable"], fig_df["n_nulos"])
    plt.xlabel("Número de valores ausentes")
    plt.ylabel("Variable")
    plt.title("Valores ausentes por variable en el dataset final para ML")
    plt.tight_layout()

    ruta_figura = FIGURAS_DIR / "nulos_por_variable_dataset_final_ML.png"
    plt.savefig(ruta_figura, dpi=300, bbox_inches="tight")
    plt.show()

    print("Figura guardada en:")
    print(ruta_figura.resolve())
else:
    print("No hay valores ausentes en el dataset final para ML.")


print("\nCONCLUSIÓN RÁPIDA:")
print(f"- El dataset final para ML tiene {df_ml.shape[0]} filas y {df_ml.shape[1]} columnas.")
print(f"- Nulos totales: {n_nulos_total} ({pct_nulos_total:.3f}%).")
print(f"- Nulos en variables biomecánicas: {n_nulos_biomecanicos} ({pct_nulos_biomecanicos:.3f}%).")

if "marca_m" in df_ml.columns:
    print(f"- Nulos en marca_m: {df_ml['marca_m'].isna().sum()}.")

if "marca_vs_nivel_cm" in df_ml.columns:
    print(f"- Nulos en marca_vs_nivel_cm: {df_ml['marca_vs_nivel_cm'].isna().sum()}.")

if "nivel_referencia_pc" in df_ml.columns:
    print(f"- Nulos en nivel_referencia_pc: {df_ml['nivel_referencia_pc'].isna().sum()}.")

print("\nDecisión metodológica:")
print("Los NaN biomecánicos se mantienen durante el EDA y se imputarán dentro del pipeline de ML.")
print("Los NaN de marca_m y marca_vs_nivel_cm asociados a saltos nulos no se imputan.")


## 6. Valid vs. foul classification

**Target:** `resultado_nulo`

Biomechanical feature packages are compared using cross-validation. The final interpretation focuses on the best temporal/spatial logistic-regression model.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    make_scorer
)

warnings.filterwarnings("ignore")


SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
DATASETS_DIR = SALIDA_DIR / "datasets"

ML_DIR = SALIDA_DIR / "ml_clasificacion_nulos_por_paquetes"
ML_TABLAS_DIR = ML_DIR / "tablas"
ML_FIGURAS_DIR = ML_DIR / "figuras"

for carpeta in [ML_DIR, ML_TABLAS_DIR, ML_FIGURAS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)


if "combinado_total_FINAL_con_resultado_y_nivel" in globals():
    df_ml = combinado_total_FINAL_con_resultado_y_nivel.copy()
    print("Usando dataset desde memoria.")
else:
    ruta_df = DATASETS_DIR / "combinado_total_FINAL_con_resultado_y_nivel.csv"

    if not ruta_df.exists():
        raise FileNotFoundError(f"No encuentro el dataset en:\n{ruta_df.resolve()}")

    df_ml = pd.read_csv(ruta_df, encoding="utf-8-sig")
    print("Dataset cargado desde:")
    print(ruta_df.resolve())

print("Dimensiones:", df_ml.shape)
print("Vídeos únicos:", df_ml["video_id"].nunique())

if len(df_ml) != df_ml["video_id"].nunique():
    raise ValueError("Hay video_id duplicados. No se debe entrenar todavía.")


df_modelo = df_ml.copy()
df_modelo = df_modelo[df_modelo["resultado_nulo"].notna()].copy()
df_modelo["resultado_nulo"] = df_modelo["resultado_nulo"].astype(int)

y = df_modelo["resultado_nulo"]

print("\nDistribución de clases:")
display(y.value_counts().rename(index={0: "valido", 1: "nulo"}))

print("\nPorcentaje de clases:")
display((y.value_counts(normalize=True) * 100).rename(index={0: "valido", 1: "nulo"}))


columnas_excluir = [
    "video_id",
    "id_salto",
    "competicion",
    "intento",
    "atleta",
    "id_atleta",
    "resultado_original",
    "marca_m",
    "resultado_nulo",
    "nivel_referencia_pc",
    "marca_vs_nivel_cm",
]

columnas_excluir = [c for c in columnas_excluir if c in df_modelo.columns]

X_total = df_modelo.drop(columns=columnas_excluir)
X_total = X_total.select_dtypes(include=[np.number]).copy()


patrones_prohibidos = [
    "marca",
    "resultado",
    "nivel",
    "atleta",
    "video",
    "competicion",
    "intento",
    "id_salto",
]

columnas_sospechosas = [
    c for c in X_total.columns
    if any(p in c.lower() for p in patrones_prohibidos)
]

if columnas_sospechosas:
    print("Columnas sospechosas eliminadas:")
    print(columnas_sospechosas)
    X_total = X_total.drop(columns=columnas_sospechosas)

print("\nTotal variables biomecánicas:", X_total.shape[1])


columnas = list(X_total.columns)

def contiene(col, patrones):
    col = col.lower()
    return any(p in col for p in patrones)


cols_angulos = [
    c for c in columnas
    if contiene(
        c,
        [
            "angulo",
            "rodilla",
            "cadera",
            "tronco",
            "inclinacion",
            "dif_",
            "diferencia"
        ]
    )
]


cols_velocidades = [
    c for c in columnas
    if contiene(c, ["velocidad", "vel_"])
    and not contiene(c, ["time", "tiempo", "delta_t"])
]


cols_temporales_espaciales = [
    c for c in columnas
    if contiene(
        c,
        [
            "distancia",
            "longitud",
            "tiempo",
            "time",
            "delta_t",
            "delta_x",
            "delta_y"
        ]
    )
]


cols_todas = columnas


cols_angulos = list(dict.fromkeys(cols_angulos))
cols_velocidades = list(dict.fromkeys(cols_velocidades))
cols_temporales_espaciales = list(dict.fromkeys(cols_temporales_espaciales))
cols_todas = list(dict.fromkeys(cols_todas))

paquetes_variables = {
    "angulos": cols_angulos,
    "velocidades": cols_velocidades,
    "temporales_espaciales": cols_temporales_espaciales,
    "todas_biomecanicas": cols_todas,
}


resumen_paquetes = []

for nombre_paquete, cols in paquetes_variables.items():
    resumen_paquetes.append({
        "paquete": nombre_paquete,
        "n_variables": len(cols),
        "variables": ", ".join(cols)
    })

resumen_paquetes_df = pd.DataFrame(resumen_paquetes)
display(resumen_paquetes_df)

resumen_paquetes_df.to_csv(
    ML_TABLAS_DIR / "resumen_paquetes_variables_clasificacion.csv",
    index=False,
    encoding="utf-8-sig"
)

for nombre_paquete, cols in paquetes_variables.items():
    pd.DataFrame({"variable": cols}).to_csv(
        ML_TABLAS_DIR / f"variables_paquete_{nombre_paquete}.csv",
        index=False,
        encoding="utf-8-sig"
    )


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_nulo": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_nulo": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_nulo": make_scorer(f1_score, pos_label=1, zero_division=0),
    "roc_auc": "roc_auc"
}


modelos = {
    "Dummy": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyClassifier(random_state=42))
        ]),
        "param_grid": {
            "model__strategy": ["most_frequent", "stratified"]
        }
    },

    "Regresion_Logistica": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                solver="liblinear",
                random_state=42
            ))
        ]),
        "param_grid": {
            "model__C": [0.01, 0.1, 1, 10],
            "model__class_weight": [None, "balanced"]
        }
    },

    "SVM": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(
                probability=True,
                random_state=42
            ))
        ]),
        "param_grid": [
            {
                "model__kernel": ["linear"],
                "model__C": [0.1, 1, 10],
                "model__class_weight": [None, "balanced"]
            },
            {
                "model__kernel": ["rbf"],
                "model__C": [0.1, 1, 10],
                "model__gamma": ["scale", 0.01, 0.1],
                "model__class_weight": [None, "balanced"]
            }
        ]
    },

    "Random_Forest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 3, 5],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2"],
            "model__class_weight": [None, "balanced"]
        }
    },

    "Gradient_Boosting": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingClassifier(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [1, 3],
            "model__subsample": [0.8, 1.0]
        }
    }
}


resultados_modelos = []
mejores_parametros = {}
reportes_clasificacion = []

for nombre_paquete, columnas_paquete in paquetes_variables.items():

    print("\n" + "#" * 80)
    print(f"PAQUETE DE VARIABLES: {nombre_paquete}")
    print(f"Número de variables: {len(columnas_paquete)}")
    print("#" * 80)

    if len(columnas_paquete) == 0:
        print(f"Paquete {nombre_paquete} vacío. Se omite.")
        continue

    X = X_total[columnas_paquete].copy()

    nulos_X = X.isna().sum().reset_index()
    nulos_X.columns = ["variable", "n_nulos"]
    nulos_X = nulos_X[nulos_X["n_nulos"] > 0].copy()
    nulos_X["porcentaje_nulos"] = nulos_X["n_nulos"] / len(X) * 100
    nulos_X["paquete"] = nombre_paquete

    display(nulos_X)

    nulos_X.to_csv(
        ML_TABLAS_DIR / f"nulos_paquete_{nombre_paquete}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    for nombre_modelo, config in modelos.items():

        print("\n" + "=" * 70)
        print(f"Entrenando modelo: {nombre_modelo} | Paquete: {nombre_paquete}")
        print("=" * 70)

        grid = GridSearchCV(
            estimator=config["pipeline"],
            param_grid=config["param_grid"],
            scoring=scoring,
            refit="f1_nulo",
            cv=cv,
            n_jobs=-1,
            return_train_score=True
        )

        grid.fit(X, y)

        clave = f"{nombre_paquete}__{nombre_modelo}"
        mejores_parametros[clave] = grid.best_params_

        cv_results = pd.DataFrame(grid.cv_results_)
        cv_results["paquete"] = nombre_paquete
        cv_results["modelo"] = nombre_modelo

        cv_results.to_csv(
            ML_TABLAS_DIR / f"cv_results_{nombre_paquete}_{nombre_modelo}.csv",
            index=False,
            encoding="utf-8-sig"
        )

        mejor_idx = grid.best_index_
        mejor_modelo = grid.best_estimator_

        y_pred_cv = cross_val_predict(
            mejor_modelo,
            X,
            y,
            cv=cv,
            n_jobs=-1
        )

        try:
            y_score_cv = cross_val_predict(
                mejor_modelo,
                X,
                y,
                cv=cv,
                method="predict_proba",
                n_jobs=-1
            )[:, 1]

            roc_auc_cv_pred = roc_auc_score(y, y_score_cv)

        except Exception:
            roc_auc_cv_pred = np.nan

        accuracy_cv_pred = accuracy_score(y, y_pred_cv)
        balanced_accuracy_cv_pred = balanced_accuracy_score(y, y_pred_cv)
        precision_nulo_cv_pred = precision_score(y, y_pred_cv, pos_label=1, zero_division=0)
        recall_nulo_cv_pred = recall_score(y, y_pred_cv, pos_label=1, zero_division=0)
        f1_nulo_cv_pred = f1_score(y, y_pred_cv, pos_label=1, zero_division=0)

        resultados_modelos.append({
            "paquete": nombre_paquete,
            "modelo": nombre_modelo,
            "n_variables": len(columnas_paquete),

            "mejor_f1_nulo_grid_mean": grid.cv_results_["mean_test_f1_nulo"][mejor_idx],
            "mejor_f1_nulo_grid_std": grid.cv_results_["std_test_f1_nulo"][mejor_idx],
            "accuracy_grid_mean": grid.cv_results_["mean_test_accuracy"][mejor_idx],
            "balanced_accuracy_grid_mean": grid.cv_results_["mean_test_balanced_accuracy"][mejor_idx],
            "precision_nulo_grid_mean": grid.cv_results_["mean_test_precision_nulo"][mejor_idx],
            "recall_nulo_grid_mean": grid.cv_results_["mean_test_recall_nulo"][mejor_idx],
            "roc_auc_grid_mean": grid.cv_results_["mean_test_roc_auc"][mejor_idx],

            "accuracy_cv_pred": accuracy_cv_pred,
            "balanced_accuracy_cv_pred": balanced_accuracy_cv_pred,
            "precision_nulo_cv_pred": precision_nulo_cv_pred,
            "recall_nulo_cv_pred": recall_nulo_cv_pred,
            "f1_nulo_cv_pred": f1_nulo_cv_pred,
            "roc_auc_cv_pred": roc_auc_cv_pred,

            "mejores_hiperparametros": json.dumps(grid.best_params_, ensure_ascii=False)
        })

        reporte = classification_report(
            y,
            y_pred_cv,
            target_names=["valido", "nulo"],
            output_dict=True,
            zero_division=0
        )

        reporte_df = pd.DataFrame(reporte).T
        reporte_df["paquete"] = nombre_paquete
        reporte_df["modelo"] = nombre_modelo

        reportes_clasificacion.append(
            reporte_df.reset_index().rename(columns={"index": "clase"})
        )

        reporte_df.to_csv(
            ML_TABLAS_DIR / f"classification_report_{nombre_paquete}_{nombre_modelo}.csv",
            index=True,
            encoding="utf-8-sig"
        )

        cm = confusion_matrix(y, y_pred_cv, labels=[0, 1])

        cm_df = pd.DataFrame(
            cm,
            index=["Real_valido", "Real_nulo"],
            columns=["Pred_valido", "Pred_nulo"]
        )

        display(cm_df)

        cm_df.to_csv(
            ML_TABLAS_DIR / f"matriz_confusion_{nombre_paquete}_{nombre_modelo}.csv",
            index=True,
            encoding="utf-8-sig"
        )

        plt.figure(figsize=(5, 4))
        plt.imshow(cm)
        plt.title(f"{nombre_modelo} - {nombre_paquete}")
        plt.xticks([0, 1], ["Pred válido", "Pred nulo"])
        plt.yticks([0, 1], ["Real válido", "Real nulo"])
        plt.colorbar(label="Número de saltos")

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, cm[i, j], ha="center", va="center")

        plt.tight_layout()
        plt.savefig(
            ML_FIGURAS_DIR / f"matriz_confusion_{nombre_paquete}_{nombre_modelo}.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

        print("Mejores hiperparámetros:")
        print(grid.best_params_)


resultados_modelos_df = pd.DataFrame(resultados_modelos)

resultados_modelos_df = resultados_modelos_df.sort_values(
    ["f1_nulo_cv_pred", "balanced_accuracy_cv_pred"],
    ascending=False
).reset_index(drop=True)

display(resultados_modelos_df)

resultados_modelos_df.to_csv(
    ML_TABLAS_DIR / "resultados_modelos_clasificacion_nulo_valido_por_paquetes.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    ML_TABLAS_DIR / "mejores_hiperparametros_clasificacion_nulo_valido_por_paquetes.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(mejores_parametros, f, ensure_ascii=False, indent=4)

reportes_clasificacion_df = pd.concat(reportes_clasificacion, ignore_index=True)

reportes_clasificacion_df.to_csv(
    ML_TABLAS_DIR / "classification_reports_todos_modelos_por_paquetes.csv",
    index=False,
    encoding="utf-8-sig"
)

display(reportes_clasificacion_df)


mejor_por_paquete = (
    resultados_modelos_df
    .sort_values(
        ["paquete", "f1_nulo_cv_pred", "balanced_accuracy_cv_pred"],
        ascending=[True, False, False]
    )
    .groupby("paquete")
    .head(1)
    .reset_index(drop=True)
)

display(mejor_por_paquete)

mejor_por_paquete.to_csv(
    ML_TABLAS_DIR / "mejor_modelo_por_paquete_clasificacion_nulos.csv",
    index=False,
    encoding="utf-8-sig"
)


plot_df = mejor_por_paquete.sort_values("f1_nulo_cv_pred", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_df["paquete"], plot_df["f1_nulo_cv_pred"])
plt.xlabel("F1-score clase nulo")
plt.ylabel("Paquete de variables")
plt.title("Mejor resultado por paquete de variables")
plt.xlim(0, 1)
plt.tight_layout()

plt.savefig(
    ML_FIGURAS_DIR / "mejor_f1_nulo_por_paquete.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plot_df = resultados_modelos_df.copy()
plot_df["modelo_paquete"] = plot_df["modelo"] + " | " + plot_df["paquete"]
plot_df = plot_df.sort_values("f1_nulo_cv_pred", ascending=True)

plt.figure(figsize=(10, max(6, 0.28 * len(plot_df))))
plt.barh(plot_df["modelo_paquete"], plot_df["f1_nulo_cv_pred"])
plt.xlabel("F1-score clase nulo")
plt.ylabel("Modelo | Paquete")
plt.title("Comparación completa de modelos y paquetes")
plt.xlim(0, 1)
plt.tight_layout()

plt.savefig(
    ML_FIGURAS_DIR / "comparacion_completa_modelos_paquetes_f1_nulo.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


mejor_global = resultados_modelos_df.iloc[0]

print("\n" + "=" * 80)
print("RESUMEN FINAL")
print("=" * 80)

print(f"Número de saltos utilizados: {len(df_modelo)}")
print(f"Saltos válidos: {(y == 0).sum()}")
print(f"Saltos nulos: {(y == 1).sum()}")

print("\nMejor combinación global:")
print(f"Paquete: {mejor_global['paquete']}")
print(f"Modelo: {mejor_global['modelo']}")
print(f"F1 nulo: {mejor_global['f1_nulo_cv_pred']:.3f}")
print(f"Balanced accuracy: {mejor_global['balanced_accuracy_cv_pred']:.3f}")

print("\nArchivos guardados en:")
print(ML_DIR.resolve())


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
    make_scorer
)

from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")


SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
DATASETS_DIR = SALIDA_DIR / "datasets"

ML_DIR = SALIDA_DIR / "ml_clasificacion_nulos_por_paquetes"
ML_TABLAS_DIR = ML_DIR / "tablas"
ML_FIGURAS_DIR = ML_DIR / "figuras"

INTERP_DIR = ML_DIR / "interpretabilidad_mejor_modelo"
INTERP_TABLAS_DIR = INTERP_DIR / "tablas"
INTERP_FIGURAS_DIR = INTERP_DIR / "figuras"

for carpeta in [INTERP_DIR, INTERP_TABLAS_DIR, INTERP_FIGURAS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)


if "combinado_total_FINAL_con_resultado_y_nivel" in globals():
    df_ml = combinado_total_FINAL_con_resultado_y_nivel.copy()
    print("Usando dataset desde memoria.")
else:
    ruta_df = DATASETS_DIR / "combinado_total_FINAL_con_resultado_y_nivel.csv"

    if not ruta_df.exists():
        raise FileNotFoundError(f"No encuentro el dataset en:\n{ruta_df.resolve()}")

    df_ml = pd.read_csv(ruta_df, encoding="utf-8-sig")
    print("Dataset cargado desde:")
    print(ruta_df.resolve())

print("Dimensiones:", df_ml.shape)
print("Vídeos únicos:", df_ml["video_id"].nunique())

if len(df_ml) != df_ml["video_id"].nunique():
    raise ValueError("Hay video_id duplicados. No se debe continuar.")


df_modelo = df_ml.copy()
df_modelo = df_modelo[df_modelo["resultado_nulo"].notna()].copy()
df_modelo["resultado_nulo"] = df_modelo["resultado_nulo"].astype(int)

y = df_modelo["resultado_nulo"]

print("\nDistribución de clases:")
display(y.value_counts().rename(index={0: "valido", 1: "nulo"}))


columnas_excluir = [
    "video_id",
    "id_salto",
    "competicion",
    "intento",
    "atleta",
    "id_atleta",
    "resultado_original",
    "marca_m",
    "resultado_nulo",
    "nivel_referencia_pc",
    "marca_vs_nivel_cm",
]

columnas_excluir = [c for c in columnas_excluir if c in df_modelo.columns]

X_total = df_modelo.drop(columns=columnas_excluir)
X_total = X_total.select_dtypes(include=[np.number]).copy()

patrones_prohibidos = [
    "marca",
    "resultado",
    "nivel",
    "atleta",
    "video",
    "competicion",
    "intento",
    "id_salto",
]

columnas_sospechosas = [
    c for c in X_total.columns
    if any(p in c.lower() for p in patrones_prohibidos)
]

if columnas_sospechosas:
    X_total = X_total.drop(columns=columnas_sospechosas)

def contiene(col, patrones):
    col = col.lower()
    return any(p in col for p in patrones)

cols_temporales_espaciales = [
    c for c in X_total.columns
    if contiene(
        c,
        [
            "distancia",
            "longitud",
            "tiempo",
            "time",
            "delta_t",
            "delta_x",
            "delta_y"
        ]
    )
]

cols_temporales_espaciales = list(dict.fromkeys(cols_temporales_espaciales))

X = X_total[cols_temporales_espaciales].copy()

print("\nVariables temporales y espaciales utilizadas:")
print("Número de variables:", X.shape[1])
display(pd.DataFrame({"variable": X.columns}))

pd.DataFrame({"variable": X.columns}).to_csv(
    INTERP_TABLAS_DIR / "variables_temporales_espaciales_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)


mejor_modelo = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=10,
        class_weight=None,
        max_iter=2000,
        solver="liblinear",
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


y_pred_cv = cross_val_predict(
    mejor_modelo,
    X,
    y,
    cv=cv,
    n_jobs=-1
)

y_prob_cv = cross_val_predict(
    mejor_modelo,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

metricas_mejor_modelo = pd.DataFrame([{
    "modelo": "Regresion_Logistica",
    "paquete": "temporales_espaciales",
    "n_variables": X.shape[1],
    "n_saltos": len(y),
    "n_validos": int((y == 0).sum()),
    "n_nulos": int((y == 1).sum()),
    "accuracy": accuracy_score(y, y_pred_cv),
    "balanced_accuracy": balanced_accuracy_score(y, y_pred_cv),
    "precision_nulo": precision_score(y, y_pred_cv, pos_label=1, zero_division=0),
    "recall_nulo": recall_score(y, y_pred_cv, pos_label=1, zero_division=0),
    "f1_nulo": f1_score(y, y_pred_cv, pos_label=1, zero_division=0),
    "roc_auc": roc_auc_score(y, y_prob_cv),
}])

display(metricas_mejor_modelo)

metricas_mejor_modelo.to_csv(
    INTERP_TABLAS_DIR / "metricas_mejor_modelo_temporales_espaciales.csv",
    index=False,
    encoding="utf-8-sig"
)


cm = confusion_matrix(y, y_pred_cv, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["Real_valido", "Real_nulo"],
    columns=["Pred_valido", "Pred_nulo"]
)

display(cm_df)

cm_df.to_csv(
    INTERP_TABLAS_DIR / "matriz_confusion_mejor_modelo.csv",
    index=True,
    encoding="utf-8-sig"
)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Matriz de confusión - Mejor modelo")
plt.xticks([0, 1], ["Pred válido", "Pred nulo"])
plt.yticks([0, 1], ["Real válido", "Real nulo"])
plt.colorbar(label="Número de saltos")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.savefig(
    INTERP_FIGURAS_DIR / "matriz_confusion_mejor_modelo.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


fpr, tpr, thresholds_roc = roc_curve(y, y_prob_cv)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Azar")
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos")
plt.title("Curva ROC - Mejor modelo")
plt.legend()
plt.tight_layout()
plt.savefig(
    INTERP_FIGURAS_DIR / "curva_roc_mejor_modelo.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y, y_prob_cv)
pr_auc = auc(recall_curve, precision_curve)

plt.figure(figsize=(6, 5))
plt.plot(recall_curve, precision_curve, label=f"AUC PR = {pr_auc:.3f}")
plt.xlabel("Recall clase nulo")
plt.ylabel("Precision clase nulo")
plt.title("Curva Precision-Recall - Mejor modelo")
plt.legend()
plt.tight_layout()
plt.savefig(
    INTERP_FIGURAS_DIR / "curva_precision_recall_mejor_modelo.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


pd.DataFrame({
    "fpr": fpr,
    "tpr": tpr,
    "threshold": thresholds_roc
}).to_csv(
    INTERP_TABLAS_DIR / "puntos_curva_roc_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "recall": recall_curve,
    "precision": precision_curve
}).to_csv(
    INTERP_TABLAS_DIR / "puntos_curva_precision_recall_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)


predicciones_video = df_modelo[[
    c for c in [
        "video_id",
        "atleta",
        "resultado_original",
        "marca_m",
        "resultado_nulo"
    ]
    if c in df_modelo.columns
]].copy()

predicciones_video["real"] = y.values
predicciones_video["prediccion"] = y_pred_cv
predicciones_video["probabilidad_nulo"] = y_prob_cv

predicciones_video["real_etiqueta"] = predicciones_video["real"].map({
    0: "valido",
    1: "nulo"
})

predicciones_video["prediccion_etiqueta"] = predicciones_video["prediccion"].map({
    0: "valido",
    1: "nulo"
})

predicciones_video["acierto"] = predicciones_video["real"] == predicciones_video["prediccion"]

def tipo_error(row):
    if row["acierto"]:
        return "acierto"
    elif row["real"] == 1 and row["prediccion"] == 0:
        return "nulo_predicho_como_valido"
    elif row["real"] == 0 and row["prediccion"] == 1:
        return "valido_predicho_como_nulo"
    else:
        return "otro"

predicciones_video["tipo_error"] = predicciones_video.apply(tipo_error, axis=1)

predicciones_video = predicciones_video.sort_values(
    ["acierto", "probabilidad_nulo"],
    ascending=[True, False]
)

display(predicciones_video)

predicciones_video.to_csv(
    INTERP_TABLAS_DIR / "predicciones_por_video_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)

errores_video = predicciones_video[predicciones_video["acierto"] == False].copy()

display(errores_video)

errores_video.to_csv(
    INTERP_TABLAS_DIR / "errores_por_video_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)


if "atleta" in predicciones_video.columns:
    errores_por_atleta = (
        predicciones_video
        .groupby("atleta")
        .agg(
            n_saltos=("video_id", "count"),
            n_errores=("acierto", lambda x: int((~x).sum())),
            n_aciertos=("acierto", lambda x: int(x.sum())),
            media_probabilidad_nulo=("probabilidad_nulo", "mean")
        )
        .reset_index()
    )

    errores_por_atleta["tasa_error"] = errores_por_atleta["n_errores"] / errores_por_atleta["n_saltos"]

    errores_por_atleta = errores_por_atleta.sort_values(
        ["n_errores", "tasa_error"],
        ascending=False
    )

    display(errores_por_atleta)

    errores_por_atleta.to_csv(
        INTERP_TABLAS_DIR / "errores_por_atleta_mejor_modelo.csv",
        index=False,
        encoding="utf-8-sig"
    )


mejor_modelo.fit(X, y)

coeficientes = mejor_modelo.named_steps["model"].coef_[0]

coeficientes_df = pd.DataFrame({
    "variable": X.columns,
    "coeficiente": coeficientes,
})

coeficientes_df["abs_coeficiente"] = coeficientes_df["coeficiente"].abs()

coeficientes_df["interpretacion"] = np.where(
    coeficientes_df["coeficiente"] > 0,
    "aumenta_probabilidad_nulo",
    "aumenta_probabilidad_valido"
)

coeficientes_df = coeficientes_df.sort_values(
    "abs_coeficiente",
    ascending=False
).reset_index(drop=True)

display(coeficientes_df)

coeficientes_df.to_csv(
    INTERP_TABLAS_DIR / "coeficientes_regresion_logistica_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)


top_coef = coeficientes_df.head(15).sort_values("coeficiente")

plt.figure(figsize=(9, max(5, 0.35 * len(top_coef))))
plt.barh(top_coef["variable"], top_coef["coeficiente"])
plt.axvline(0, linestyle="--")
plt.xlabel("Coeficiente estandarizado")
plt.ylabel("Variable")
plt.title("Coeficientes principales - Regresión logística")
plt.tight_layout()
plt.savefig(
    INTERP_FIGURAS_DIR / "coeficientes_principales_regresion_logistica.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


scorer_f1_nulo = make_scorer(
    f1_score,
    pos_label=1,
    zero_division=0
)

importancias_folds = []

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    modelo_fold = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=10,
            class_weight=None,
            max_iter=2000,
            solver="liblinear",
            random_state=42
        ))
    ])

    modelo_fold.fit(X_train, y_train)

    perm = permutation_importance(
        modelo_fold,
        X_test,
        y_test,
        scoring=scorer_f1_nulo,
        n_repeats=30,
        random_state=42,
        n_jobs=-1
    )

    fold_df = pd.DataFrame({
        "variable": X.columns,
        "importancia_media_fold": perm.importances_mean,
        "importancia_std_fold": perm.importances_std,
        "fold": fold_idx
    })

    importancias_folds.append(fold_df)

importancias_folds_df = pd.concat(importancias_folds, ignore_index=True)

importancias_folds_df.to_csv(
    INTERP_TABLAS_DIR / "permutation_importance_folds_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)

importancia_perm_df = (
    importancias_folds_df
    .groupby("variable")
    .agg(
        importancia_media=("importancia_media_fold", "mean"),
        importancia_std=("importancia_media_fold", "std")
    )
    .reset_index()
)

importancia_perm_df = importancia_perm_df.sort_values(
    "importancia_media",
    ascending=False
).reset_index(drop=True)

display(importancia_perm_df)

importancia_perm_df.to_csv(
    INTERP_TABLAS_DIR / "permutation_importance_resumen_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)

top_perm = importancia_perm_df.head(15).sort_values("importancia_media")

plt.figure(figsize=(9, max(5, 0.35 * len(top_perm))))
plt.barh(top_perm["variable"], top_perm["importancia_media"])
plt.xlabel("Importancia media por permutación")
plt.ylabel("Variable")
plt.title("Permutation importance - Mejor modelo")
plt.tight_layout()
plt.savefig(
    INTERP_FIGURAS_DIR / "permutation_importance_mejor_modelo.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


resumen_interpretabilidad = pd.DataFrame([{
    "modelo": "Regresión Logística",
    "paquete_variables": "temporales_espaciales",
    "n_variables": X.shape[1],
    "accuracy": accuracy_score(y, y_pred_cv),
    "balanced_accuracy": balanced_accuracy_score(y, y_pred_cv),
    "precision_nulo": precision_score(y, y_pred_cv, pos_label=1, zero_division=0),
    "recall_nulo": recall_score(y, y_pred_cv, pos_label=1, zero_division=0),
    "f1_nulo": f1_score(y, y_pred_cv, pos_label=1, zero_division=0),
    "roc_auc": roc_auc_score(y, y_prob_cv),
    "n_errores": int((y != y_pred_cv).sum()),
    "n_aciertos": int((y == y_pred_cv).sum()),
    "n_nulos_detectados": int(((y == 1) & (y_pred_cv == 1)).sum()),
    "n_nulos_no_detectados": int(((y == 1) & (y_pred_cv == 0)).sum()),
    "n_validos_correctos": int(((y == 0) & (y_pred_cv == 0)).sum()),
    "n_validos_predichos_nulos": int(((y == 0) & (y_pred_cv == 1)).sum()),
}])

display(resumen_interpretabilidad)

resumen_interpretabilidad.to_csv(
    INTERP_TABLAS_DIR / "resumen_interpretabilidad_mejor_modelo.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 80)
print("RESUMEN INTERPRETABILIDAD MEJOR MODELO")
print("=" * 80)

print("Modelo: Regresión Logística")
print("Paquete: temporales_espaciales")
print(f"Número de variables: {X.shape[1]}")
print(f"F1 clase nulo: {f1_score(y, y_pred_cv, pos_label=1, zero_division=0):.3f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y, y_pred_cv):.3f}")
print(f"ROC-AUC: {roc_auc_score(y, y_prob_cv):.3f}")

print("\nCoeficientes principales guardados en:")
print((INTERP_TABLAS_DIR / "coeficientes_regresion_logistica_mejor_modelo.csv").resolve())

print("\nPermutation importance guardada en:")
print((INTERP_TABLAS_DIR / "permutation_importance_resumen_mejor_modelo.csv").resolve())

print("\nPredicciones y errores por vídeo guardados en:")
print((INTERP_TABLAS_DIR / "predicciones_por_video_mejor_modelo.csv").resolve())
print((INTERP_TABLAS_DIR / "errores_por_video_mejor_modelo.csv").resolve())

print("\nCarpeta completa:")
print(INTERP_DIR.resolve())


## 7. Absolute jump-distance regression

**Target:** jump distance in centimetres for valid attempts.

The models are compared against a dummy baseline to test whether video-derived variables contain useful information about absolute performance.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import json

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")


SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
DATASETS_DIR = SALIDA_DIR / "datasets"

OUT_DIR = SALIDA_DIR / "ml_regresion_marca_absoluta_rapida"
TABLAS_DIR = OUT_DIR / "tablas"
FIGURAS_DIR = OUT_DIR / "figuras"

for carpeta in [OUT_DIR, TABLAS_DIR, FIGURAS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

if "combinado_total_FINAL_con_resultado_y_nivel" in globals():
    df = combinado_total_FINAL_con_resultado_y_nivel.copy()
    print("Usando dataset desde memoria.")
else:
    ruta = DATASETS_DIR / "combinado_total_FINAL_con_resultado_y_nivel.csv"
    df = pd.read_csv(ruta, encoding="utf-8-sig")
    print("Dataset cargado desde:", ruta.resolve())

print("Dimensiones:", df.shape)


df["marca_m"] = pd.to_numeric(df["marca_m"], errors="coerce")

df_abs = df[
    (df["resultado_nulo"] == 0) &
    (df["marca_m"].notna())
].copy()

df_abs["marca_cm"] = df_abs["marca_m"] * 100

y = df_abs["marca_cm"].astype(float)

print("\nDataset para marca absoluta:")
print("N saltos:", len(df_abs))
print("Media marca:", y.mean())
print("Mediana marca:", y.median())
print("STD marca:", y.std())
print("Mínimo:", y.min())
print("Máximo:", y.max())

display(pd.DataFrame([{
    "n_saltos": len(df_abs),
    "media_marca_cm": y.mean(),
    "mediana_marca_cm": y.median(),
    "std_marca_cm": y.std(),
    "min_marca_cm": y.min(),
    "max_marca_cm": y.max()
}]))


columnas_excluir = [
    "video_id",
    "id_salto",
    "competicion",
    "intento",
    "atleta",
    "id_atleta",
    "resultado_original",
    "marca_m",
    "marca_cm",
    "resultado_nulo",
    "nivel_referencia_pc",
    "marca_vs_nivel_cm",
]

columnas_excluir = [c for c in columnas_excluir if c in df_abs.columns]

X_total = df_abs.drop(columns=columnas_excluir)
X_total = X_total.select_dtypes(include=[np.number]).copy()


patrones_prohibidos = [
    "marca",
    "resultado",
    "nivel",
    "atleta",
    "video",
    "competicion",
    "intento",
    "id_salto"
]

cols_sospechosas = [
    c for c in X_total.columns
    if any(p in c.lower() for p in patrones_prohibidos)
]

if cols_sospechosas:
    print("Columnas sospechosas eliminadas:", cols_sospechosas)
    X_total = X_total.drop(columns=cols_sospechosas)

print("\nVariables biomecánicas totales:", X_total.shape[1])


def contiene(col, patrones):
    col = col.lower()
    return any(p in col for p in patrones)

columnas = list(X_total.columns)

cols_angulos = [
    c for c in columnas
    if contiene(c, [
        "angulo",
        "rodilla",
        "cadera",
        "tronco",
        "inclinacion",
        "dif_",
        "diferencia"
    ])
]

cols_velocidades = [
    c for c in columnas
    if contiene(c, ["velocidad", "vel_"])
    and not contiene(c, ["time", "tiempo", "delta_t"])
]

cols_temporales_espaciales = [
    c for c in columnas
    if contiene(c, [
        "distancia",
        "longitud",
        "tiempo",
        "time",
        "delta_t",
        "delta_x",
        "delta_y"
    ])
]

cols_velocidad_mas_temporales = list(dict.fromkeys(
    cols_velocidades + cols_temporales_espaciales
))

paquetes = {
    "angulos": list(dict.fromkeys(cols_angulos)),
    "velocidades": list(dict.fromkeys(cols_velocidades)),
    "temporales_espaciales": list(dict.fromkeys(cols_temporales_espaciales)),
    "velocidades_mas_temporales": cols_velocidad_mas_temporales,
    "todas_biomecanicas": list(dict.fromkeys(columnas)),
}

resumen_paquetes = pd.DataFrame([
    {
        "paquete": nombre,
        "n_variables": len(cols),
        "variables": ", ".join(cols)
    }
    for nombre, cols in paquetes.items()
])

display(resumen_paquetes)

resumen_paquetes.to_csv(
    TABLAS_DIR / "resumen_paquetes_marca_absoluta.csv",
    index=False,
    encoding="utf-8-sig"
)


cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

modelos = {
    "Dummy": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyRegressor())
        ]),
        "param_grid": {
            "model__strategy": ["mean", "median"]
        }
    },

    "Ridge": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(random_state=42))
        ]),
        "param_grid": {
            "model__alpha": [1, 10, 100]
        }
    },

    "ElasticNet": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(max_iter=5000, random_state=42))
        ]),
        "param_grid": {
            "model__alpha": [0.1, 1, 10],
            "model__l1_ratio": [0.1, 0.5, 0.9]
        }
    },

    "SVR": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR())
        ]),
        "param_grid": {
            "model__kernel": ["linear", "rbf"],
            "model__C": [0.1, 1, 10],
            "model__epsilon": [5, 10, 20],
            "model__gamma": ["scale"]
        }
    },

    "Random_Forest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 3, 5],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", 1.0]
        }
    },

    "Gradient_Boosting": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [1, 3],
            "model__subsample": [0.8, 1.0]
        }
    }
}

scoring = {
    "neg_mae": "neg_mean_absolute_error",
    "neg_rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}


resultados = []
predicciones = []

for nombre_paquete, cols in paquetes.items():

    if len(cols) == 0:
        continue

    X = X_total[cols].copy()

    print("\n" + "#" * 80)
    print(f"PAQUETE: {nombre_paquete} | Variables: {len(cols)}")
    print("#" * 80)

    for nombre_modelo, config in modelos.items():

        print(f"Entrenando {nombre_modelo}...")

        grid = GridSearchCV(
            estimator=config["pipeline"],
            param_grid=config["param_grid"],
            scoring=scoring,
            refit="neg_mae",
            cv=cv,
            n_jobs=-1,
            return_train_score=True
        )

        grid.fit(X, y)

        mejor_modelo = grid.best_estimator_

        y_pred_cv = cross_val_predict(
            mejor_modelo,
            X,
            y,
            cv=cv,
            n_jobs=-1
        )

        mae = mean_absolute_error(y, y_pred_cv)
        rmse = np.sqrt(mean_squared_error(y, y_pred_cv))
        r2 = r2_score(y, y_pred_cv)

        resultados.append({
            "paquete": nombre_paquete,
            "modelo": nombre_modelo,
            "n_variables": len(cols),
            "n_saltos": len(y),
            "mae_cv_pred_cm": mae,
            "rmse_cv_pred_cm": rmse,
            "r2_cv_pred": r2,
            "mejores_hiperparametros": json.dumps(grid.best_params_, ensure_ascii=False)
        })

        pred_tmp = df_abs[[
            c for c in [
                "video_id",
                "atleta",
                "resultado_original",
                "marca_m",
                "marca_cm",
                "nivel_referencia_pc",
                "marca_vs_nivel_cm"
            ]
            if c in df_abs.columns
        ]].copy()

        pred_tmp["paquete"] = nombre_paquete
        pred_tmp["modelo"] = nombre_modelo
        pred_tmp["y_real_cm"] = y.values
        pred_tmp["y_pred_cm"] = y_pred_cv
        pred_tmp["error_cm"] = pred_tmp["y_pred_cm"] - pred_tmp["y_real_cm"]
        pred_tmp["abs_error_cm"] = pred_tmp["error_cm"].abs()

        predicciones.append(pred_tmp)

        print(f"MAE: {mae:.2f} cm | RMSE: {rmse:.2f} cm | R2: {r2:.3f}")


resultados_df = pd.DataFrame(resultados)

resultados_df = resultados_df.sort_values(
    ["mae_cv_pred_cm", "rmse_cv_pred_cm"],
    ascending=[True, True]
).reset_index(drop=True)

display(resultados_df)

resultados_df.to_csv(
    TABLAS_DIR / "resultados_regresion_marca_absoluta.csv",
    index=False,
    encoding="utf-8-sig"
)

predicciones_df = pd.concat(predicciones, ignore_index=True)

predicciones_df.to_csv(
    TABLAS_DIR / "predicciones_regresion_marca_absoluta.csv",
    index=False,
    encoding="utf-8-sig"
)


mejor_por_paquete = (
    resultados_df
    .sort_values(["paquete", "mae_cv_pred_cm", "rmse_cv_pred_cm"])
    .groupby("paquete")
    .head(1)
    .reset_index(drop=True)
)

display(mejor_por_paquete)

mejor_por_paquete.to_csv(
    TABLAS_DIR / "mejor_modelo_por_paquete_marca_absoluta.csv",
    index=False,
    encoding="utf-8-sig"
)


mejor_global = resultados_df.iloc[0]

print("\n" + "=" * 80)
print("MEJOR MODELO GLOBAL - MARCA ABSOLUTA")
print("=" * 80)
print(f"Paquete: {mejor_global['paquete']}")
print(f"Modelo: {mejor_global['modelo']}")
print(f"MAE: {mejor_global['mae_cv_pred_cm']:.2f} cm")
print(f"RMSE: {mejor_global['rmse_cv_pred_cm']:.2f} cm")
print(f"R²: {mejor_global['r2_cv_pred']:.3f}")
print("Hiperparámetros:")
print(mejor_global["mejores_hiperparametros"])


plot_df = mejor_por_paquete.sort_values("mae_cv_pred_cm", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_df["paquete"], plot_df["mae_cv_pred_cm"])
plt.xlabel("MAE medio absoluto (cm)")
plt.ylabel("Paquete")
plt.title("Mejor MAE por paquete - Marca absoluta")
plt.tight_layout()
plt.savefig(
    FIGURAS_DIR / "mejor_mae_por_paquete_marca_absoluta.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

pred_mejor = predicciones_df[
    (predicciones_df["paquete"] == mejor_global["paquete"]) &
    (predicciones_df["modelo"] == mejor_global["modelo"])
].copy()

pred_mejor = pred_mejor.sort_values("abs_error_cm", ascending=False)

display(pred_mejor)

pred_mejor.to_csv(
    TABLAS_DIR / "predicciones_mejor_modelo_marca_absoluta.csv",
    index=False,
    encoding="utf-8-sig"
)

plt.figure(figsize=(6, 6))
plt.scatter(pred_mejor["y_real_cm"], pred_mejor["y_pred_cm"])

min_val = min(pred_mejor["y_real_cm"].min(), pred_mejor["y_pred_cm"].min())
max_val = max(pred_mejor["y_real_cm"].max(), pred_mejor["y_pred_cm"].max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Marca real (cm)")
plt.ylabel("Marca predicha (cm)")
plt.title("Marca absoluta: real vs predicho")
plt.tight_layout()
plt.savefig(
    FIGURAS_DIR / "real_vs_predicho_marca_absoluta.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("\nArchivos guardados en:")
print(OUT_DIR.resolve())


## 8. Performance relative to athlete level

**Target:** `marca_vs_nivel_cm`, the difference between the attempt distance and the athlete's estimated previous level.

The analysis includes a sensitivity study without extreme target values and an interpretation of the strongest model.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    make_scorer
)

warnings.filterwarnings("ignore")


SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
DATASETS_DIR = SALIDA_DIR / "datasets"

ML_DIR = SALIDA_DIR / "ml_regresion_rendimiento_relativo"
ML_TABLAS_DIR = ML_DIR / "tablas"
ML_FIGURAS_DIR = ML_DIR / "figuras"

for carpeta in [ML_DIR, ML_TABLAS_DIR, ML_FIGURAS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)


if "combinado_total_FINAL_con_resultado_y_nivel" in globals():
    df_ml = combinado_total_FINAL_con_resultado_y_nivel.copy()
    print("Usando dataset desde memoria.")
else:
    ruta_df = DATASETS_DIR / "combinado_total_FINAL_con_resultado_y_nivel.csv"

    if not ruta_df.exists():
        raise FileNotFoundError(f"No encuentro el dataset en:\n{ruta_df.resolve()}")

    df_ml = pd.read_csv(ruta_df, encoding="utf-8-sig")
    print("Dataset cargado desde:")
    print(ruta_df.resolve())

print("Dimensiones dataset completo:", df_ml.shape)
print("Vídeos únicos:", df_ml["video_id"].nunique())

if len(df_ml) != df_ml["video_id"].nunique():
    raise ValueError("Hay video_id duplicados. No se debe entrenar todavía.")


columnas_necesarias = ["resultado_nulo", "marca_vs_nivel_cm"]

for col in columnas_necesarias:
    if col not in df_ml.columns:
        raise ValueError(f"Falta la columna necesaria: {col}")

df_reg = df_ml.copy()

df_reg = df_reg[
    (df_reg["resultado_nulo"] == 0) &
    (df_reg["marca_vs_nivel_cm"].notna())
].copy()

y = df_reg["marca_vs_nivel_cm"].astype(float)

print("\nDataset para regresión:")
print("Saltos válidos con target disponible:", len(df_reg))
print("Media target:", y.mean())
print("Mediana target:", y.median())
print("Desviación típica target:", y.std())
print("Mínimo target:", y.min())
print("Máximo target:", y.max())

resumen_target = pd.DataFrame([{
    "n_saltos_regresion": len(df_reg),
    "media_marca_vs_nivel_cm": y.mean(),
    "mediana_marca_vs_nivel_cm": y.median(),
    "std_marca_vs_nivel_cm": y.std(),
    "min_marca_vs_nivel_cm": y.min(),
    "max_marca_vs_nivel_cm": y.max()
}])

display(resumen_target)

resumen_target.to_csv(
    ML_TABLAS_DIR / "resumen_target_regresion_marca_vs_nivel_cm.csv",
    index=False,
    encoding="utf-8-sig"
)


plt.figure(figsize=(7, 5))
plt.hist(y, bins=10, edgecolor="black")
plt.axvline(y.mean(), linestyle="--", label=f"Media = {y.mean():.2f} cm")
plt.axvline(y.median(), linestyle=":", label=f"Mediana = {y.median():.2f} cm")
plt.xlabel("Marca respecto al nivel del atleta (cm)")
plt.ylabel("Número de saltos")
plt.title("Distribución del rendimiento relativo")
plt.legend()
plt.tight_layout()

plt.savefig(
    ML_FIGURAS_DIR / "distribucion_target_marca_vs_nivel_cm.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


columnas_excluir = [
    "video_id",
    "id_salto",
    "competicion",
    "intento",
    "atleta",
    "id_atleta",
    "resultado_original",
    "marca_m",
    "resultado_nulo",
    "nivel_referencia_pc",
    "marca_vs_nivel_cm",
]

columnas_excluir = [c for c in columnas_excluir if c in df_reg.columns]

X_total = df_reg.drop(columns=columnas_excluir)
X_total = X_total.select_dtypes(include=[np.number]).copy()


patrones_prohibidos = [
    "marca",
    "resultado",
    "nivel",
    "atleta",
    "video",
    "competicion",
    "intento",
    "id_salto",
]

columnas_sospechosas = [
    c for c in X_total.columns
    if any(p in c.lower() for p in patrones_prohibidos)
]

if columnas_sospechosas:
    print("Columnas sospechosas eliminadas:")
    print(columnas_sospechosas)
    X_total = X_total.drop(columns=columnas_sospechosas)

print("\nTotal variables biomecánicas:", X_total.shape[1])

pd.DataFrame({"variable": X_total.columns}).to_csv(
    ML_TABLAS_DIR / "variables_biomecanicas_disponibles_regresion.csv",
    index=False,
    encoding="utf-8-sig"
)


columnas = list(X_total.columns)

def contiene(col, patrones):
    col = col.lower()
    return any(p in col for p in patrones)


cols_angulos = [
    c for c in columnas
    if contiene(
        c,
        [
            "angulo",
            "rodilla",
            "cadera",
            "tronco",
            "inclinacion",
            "dif_",
            "diferencia"
        ]
    )
]


cols_velocidades = [
    c for c in columnas
    if contiene(c, ["velocidad", "vel_"])
    and not contiene(c, ["time", "tiempo", "delta_t"])
]


cols_temporales_espaciales = [
    c for c in columnas
    if contiene(
        c,
        [
            "distancia",
            "longitud",
            "tiempo",
            "time",
            "delta_t",
            "delta_x",
            "delta_y"
        ]
    )
]


cols_todas = columnas


cols_angulos = list(dict.fromkeys(cols_angulos))
cols_velocidades = list(dict.fromkeys(cols_velocidades))
cols_temporales_espaciales = list(dict.fromkeys(cols_temporales_espaciales))
cols_todas = list(dict.fromkeys(cols_todas))

paquetes_variables = {
    "angulos": cols_angulos,
    "velocidades": cols_velocidades,
    "temporales_espaciales": cols_temporales_espaciales,
    "todas_biomecanicas": cols_todas,
}

resumen_paquetes = []

for nombre_paquete, cols in paquetes_variables.items():
    resumen_paquetes.append({
        "paquete": nombre_paquete,
        "n_variables": len(cols),
        "variables": ", ".join(cols)
    })

resumen_paquetes_df = pd.DataFrame(resumen_paquetes)

display(resumen_paquetes_df)

resumen_paquetes_df.to_csv(
    ML_TABLAS_DIR / "resumen_paquetes_variables_regresion.csv",
    index=False,
    encoding="utf-8-sig"
)

for nombre_paquete, cols in paquetes_variables.items():
    pd.DataFrame({"variable": cols}).to_csv(
        ML_TABLAS_DIR / f"variables_paquete_{nombre_paquete}_regresion.csv",
        index=False,
        encoding="utf-8-sig"
    )


cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "neg_mae": "neg_mean_absolute_error",
    "neg_rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}


modelos = {
    "Dummy": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyRegressor())
        ]),
        "param_grid": {
            "model__strategy": ["mean", "median"]
        }
    },

    "Ridge": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(random_state=42))
        ]),
        "param_grid": {
            "model__alpha": [0.1, 1, 10, 100]
        }
    },

    "ElasticNet": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(
                max_iter=5000,
                random_state=42
            ))
        ]),
        "param_grid": {
            "model__alpha": [0.01, 0.1, 1, 10],
            "model__l1_ratio": [0.1, 0.5, 0.9]
        }
    },

    "SVR": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR())
        ]),
        "param_grid": {
            "model__kernel": ["linear", "rbf"],
            "model__C": [0.1, 1, 10],
            "model__epsilon": [1, 5, 10],
            "model__gamma": ["scale"]
        }
    },

    "Random_Forest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 3, 5],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", 1.0]
        }
    },

    "Gradient_Boosting": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [1, 3],
            "model__subsample": [0.8, 1.0]
        }
    }
}


resultados_modelos = []
mejores_parametros = []
predicciones_todos_modelos = []

for nombre_paquete, columnas_paquete in paquetes_variables.items():

    print("\n" + "#" * 80)
    print(f"PAQUETE DE VARIABLES: {nombre_paquete}")
    print(f"Número de variables: {len(columnas_paquete)}")
    print("#" * 80)

    if len(columnas_paquete) == 0:
        print(f"Paquete {nombre_paquete} vacío. Se omite.")
        continue

    X = X_total[columnas_paquete].copy()

    nulos_X = X.isna().sum().reset_index()
    nulos_X.columns = ["variable", "n_nulos"]
    nulos_X = nulos_X[nulos_X["n_nulos"] > 0].copy()
    nulos_X["porcentaje_nulos"] = nulos_X["n_nulos"] / len(X) * 100
    nulos_X["paquete"] = nombre_paquete

    display(nulos_X)

    nulos_X.to_csv(
        ML_TABLAS_DIR / f"nulos_paquete_{nombre_paquete}_regresion.csv",
        index=False,
        encoding="utf-8-sig"
    )

    for nombre_modelo, config in modelos.items():

        print("\n" + "=" * 70)
        print(f"Entrenando modelo: {nombre_modelo} | Paquete: {nombre_paquete}")
        print("=" * 70)

        grid = GridSearchCV(
            estimator=config["pipeline"],
            param_grid=config["param_grid"],
            scoring=scoring,
            refit="neg_mae",
            cv=cv,
            n_jobs=-1,
            return_train_score=True
        )

        grid.fit(X, y)

        cv_results = pd.DataFrame(grid.cv_results_)
        cv_results["paquete"] = nombre_paquete
        cv_results["modelo"] = nombre_modelo

        cv_results.to_csv(
            ML_TABLAS_DIR / f"cv_results_regresion_{nombre_paquete}_{nombre_modelo}.csv",
            index=False,
            encoding="utf-8-sig"
        )

        mejor_idx = grid.best_index_
        mejor_modelo = grid.best_estimator_

        y_pred_cv = cross_val_predict(
            mejor_modelo,
            X,
            y,
            cv=cv,
            n_jobs=-1
        )

        mae = mean_absolute_error(y, y_pred_cv)
        rmse = np.sqrt(mean_squared_error(y, y_pred_cv))
        r2 = r2_score(y, y_pred_cv)

        resultados_modelos.append({
            "paquete": nombre_paquete,
            "modelo": nombre_modelo,
            "n_variables": len(columnas_paquete),
            "n_saltos": len(y),

            "mae_cv_pred_cm": mae,
            "rmse_cv_pred_cm": rmse,
            "r2_cv_pred": r2,

            "mae_grid_mean_cm": -grid.cv_results_["mean_test_neg_mae"][mejor_idx],
            "mae_grid_std_cm": grid.cv_results_["std_test_neg_mae"][mejor_idx],
            "rmse_grid_mean_cm": -grid.cv_results_["mean_test_neg_rmse"][mejor_idx],
            "r2_grid_mean": grid.cv_results_["mean_test_r2"][mejor_idx],

            "mejores_hiperparametros": json.dumps(grid.best_params_, ensure_ascii=False)
        })

        mejores_parametros.append({
            "paquete": nombre_paquete,
            "modelo": nombre_modelo,
            "mejores_hiperparametros": json.dumps(grid.best_params_, ensure_ascii=False)
        })

        pred_tmp = df_reg[[
            c for c in [
                "video_id",
                "atleta",
                "resultado_original",
                "marca_m",
                "nivel_referencia_pc",
                "marca_vs_nivel_cm"
            ]
            if c in df_reg.columns
        ]].copy()

        pred_tmp["paquete"] = nombre_paquete
        pred_tmp["modelo"] = nombre_modelo
        pred_tmp["y_real_cm"] = y.values
        pred_tmp["y_pred_cm"] = y_pred_cv
        pred_tmp["error_cm"] = pred_tmp["y_pred_cm"] - pred_tmp["y_real_cm"]
        pred_tmp["abs_error_cm"] = pred_tmp["error_cm"].abs()

        predicciones_todos_modelos.append(pred_tmp)

        print(f"MAE: {mae:.3f} cm")
        print(f"RMSE: {rmse:.3f} cm")
        print(f"R2: {r2:.3f}")
        print("Mejores hiperparámetros:")
        print(grid.best_params_)


resultados_modelos_df = pd.DataFrame(resultados_modelos)

resultados_modelos_df = resultados_modelos_df.sort_values(
    ["mae_cv_pred_cm", "rmse_cv_pred_cm"],
    ascending=[True, True]
).reset_index(drop=True)

display(resultados_modelos_df)

resultados_modelos_df.to_csv(
    ML_TABLAS_DIR / "resultados_modelos_regresion_marca_vs_nivel_por_paquetes.csv",
    index=False,
    encoding="utf-8-sig"
)

mejores_parametros_df = pd.DataFrame(mejores_parametros)

mejores_parametros_df.to_csv(
    ML_TABLAS_DIR / "mejores_hiperparametros_regresion_marca_vs_nivel_por_paquetes.csv",
    index=False,
    encoding="utf-8-sig"
)

predicciones_todos_modelos_df = pd.concat(predicciones_todos_modelos, ignore_index=True)

predicciones_todos_modelos_df.to_csv(
    ML_TABLAS_DIR / "predicciones_todos_modelos_regresion.csv",
    index=False,
    encoding="utf-8-sig"
)

display(predicciones_todos_modelos_df.head())


mejor_por_paquete = (
    resultados_modelos_df
    .sort_values(
        ["paquete", "mae_cv_pred_cm", "rmse_cv_pred_cm"],
        ascending=[True, True, True]
    )
    .groupby("paquete")
    .head(1)
    .reset_index(drop=True)
)

display(mejor_por_paquete)

mejor_por_paquete.to_csv(
    ML_TABLAS_DIR / "mejor_modelo_por_paquete_regresion_marca_vs_nivel.csv",
    index=False,
    encoding="utf-8-sig"
)


plot_df = mejor_por_paquete.sort_values("mae_cv_pred_cm", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_df["paquete"], plot_df["mae_cv_pred_cm"])
plt.xlabel("MAE medio absoluto (cm)")
plt.ylabel("Paquete de variables")
plt.title("Mejor MAE por paquete de variables")
plt.tight_layout()

plt.savefig(
    ML_FIGURAS_DIR / "mejor_mae_por_paquete_regresion.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


plot_df = resultados_modelos_df.copy()
plot_df["modelo_paquete"] = plot_df["modelo"] + " | " + plot_df["paquete"]
plot_df = plot_df.sort_values("mae_cv_pred_cm", ascending=False)

plt.figure(figsize=(10, max(6, 0.28 * len(plot_df))))
plt.barh(plot_df["modelo_paquete"], plot_df["mae_cv_pred_cm"])
plt.xlabel("MAE medio absoluto (cm)")
plt.ylabel("Modelo | Paquete")
plt.title("Comparación completa de modelos y paquetes - Regresión")
plt.tight_layout()

plt.savefig(
    ML_FIGURAS_DIR / "comparacion_completa_modelos_paquetes_mae_regresion.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


mejor_global = resultados_modelos_df.iloc[0]

print("\n" + "=" * 80)
print("MEJOR MODELO GLOBAL DE REGRESIÓN")
print("=" * 80)
print(f"Paquete: {mejor_global['paquete']}")
print(f"Modelo: {mejor_global['modelo']}")
print(f"MAE: {mejor_global['mae_cv_pred_cm']:.3f} cm")
print(f"RMSE: {mejor_global['rmse_cv_pred_cm']:.3f} cm")
print(f"R2: {mejor_global['r2_cv_pred']:.3f}")
print("Hiperparámetros:")
print(mejor_global["mejores_hiperparametros"])

pred_mejor = predicciones_todos_modelos_df[
    (predicciones_todos_modelos_df["paquete"] == mejor_global["paquete"]) &
    (predicciones_todos_modelos_df["modelo"] == mejor_global["modelo"])
].copy()

pred_mejor = pred_mejor.sort_values("abs_error_cm", ascending=False)

display(pred_mejor)

pred_mejor.to_csv(
    ML_TABLAS_DIR / "predicciones_mejor_modelo_regresion.csv",
    index=False,
    encoding="utf-8-sig"
)


plt.figure(figsize=(6, 6))
plt.scatter(pred_mejor["y_real_cm"], pred_mejor["y_pred_cm"])
min_val = min(pred_mejor["y_real_cm"].min(), pred_mejor["y_pred_cm"].min())
max_val = max(pred_mejor["y_real_cm"].max(), pred_mejor["y_pred_cm"].max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Valor real marca_vs_nivel_cm")
plt.ylabel("Predicción marca_vs_nivel_cm")
plt.title("Valores reales vs predichos - Mejor modelo")
plt.tight_layout()

plt.savefig(
    ML_FIGURAS_DIR / "real_vs_predicho_mejor_modelo_regresion.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


plt.figure(figsize=(8, 5))
plt.hist(pred_mejor["error_cm"], bins=10, edgecolor="black")
plt.axvline(0, linestyle="--")
plt.xlabel("Error de predicción (cm)")
plt.ylabel("Número de saltos")
plt.title("Distribución de errores - Mejor modelo de regresión")
plt.tight_layout()

plt.savefig(
    ML_FIGURAS_DIR / "distribucion_errores_mejor_modelo_regresion.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


top_errores = pred_mejor.head(15).sort_values("abs_error_cm", ascending=True)

plt.figure(figsize=(9, max(5, 0.35 * len(top_errores))))
plt.barh(top_errores["video_id"], top_errores["abs_error_cm"])
plt.xlabel("Error absoluto (cm)")
plt.ylabel("Video")
plt.title("Mayores errores absolutos - Mejor modelo")
plt.tight_layout()

plt.savefig(
    ML_FIGURAS_DIR / "mayores_errores_absolutos_mejor_modelo_regresion.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\n" + "=" * 80)
print("RESUMEN FINAL REGRESIÓN")
print("=" * 80)

print(f"Número de saltos utilizados: {len(df_reg)}")
print(f"Target: marca_vs_nivel_cm")
print(f"Media del target: {y.mean():.2f} cm")
print(f"Desviación típica del target: {y.std():.2f} cm")

print("\nMejor combinación global:")
print(f"Paquete: {mejor_global['paquete']}")
print(f"Modelo: {mejor_global['modelo']}")
print(f"MAE: {mejor_global['mae_cv_pred_cm']:.2f} cm")
print(f"RMSE: {mejor_global['rmse_cv_pred_cm']:.2f} cm")
print(f"R2: {mejor_global['r2_cv_pred']:.3f}")

print("\nArchivos guardados en:")
print(ML_DIR.resolve())


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")


SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
DATASETS_DIR = SALIDA_DIR / "datasets"

SENS_DIR = SALIDA_DIR / "ml_regresion_rendimiento_relativo_sensibilidad"
SENS_TABLAS_DIR = SENS_DIR / "tablas"
SENS_FIGURAS_DIR = SENS_DIR / "figuras"

for carpeta in [SENS_DIR, SENS_TABLAS_DIR, SENS_FIGURAS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)


if "combinado_total_FINAL_con_resultado_y_nivel" in globals():
    df_ml = combinado_total_FINAL_con_resultado_y_nivel.copy()
    print("Usando dataset desde memoria.")
else:
    ruta_df = DATASETS_DIR / "combinado_total_FINAL_con_resultado_y_nivel.csv"

    if not ruta_df.exists():
        raise FileNotFoundError(f"No encuentro el dataset en:\n{ruta_df.resolve()}")

    df_ml = pd.read_csv(ruta_df, encoding="utf-8-sig")
    print("Dataset cargado desde:")
    print(ruta_df.resolve())

print("Dimensiones dataset completo:", df_ml.shape)
print("Vídeos únicos:", df_ml["video_id"].nunique())

if len(df_ml) != df_ml["video_id"].nunique():
    raise ValueError("Hay video_id duplicados. No se debe continuar.")


df_reg = df_ml[
    (df_ml["resultado_nulo"] == 0) &
    (df_ml["marca_vs_nivel_cm"].notna())
].copy()

df_reg["marca_vs_nivel_cm"] = df_reg["marca_vs_nivel_cm"].astype(float)

print("\nDataset completo de regresión:")
print("Saltos válidos con target disponible:", len(df_reg))


y_target = df_reg["marca_vs_nivel_cm"]


q1 = y_target.quantile(0.25)
q3 = y_target.quantile(0.75)
iqr = q3 - q1

limite_inferior_iqr = q1 - 1.5 * iqr
limite_superior_iqr = q3 + 1.5 * iqr

df_reg["outlier_iqr_target"] = (
    (df_reg["marca_vs_nivel_cm"] < limite_inferior_iqr) |
    (df_reg["marca_vs_nivel_cm"] > limite_superior_iqr)
)


mediana = y_target.median()
mad = np.median(np.abs(y_target - mediana))

if mad == 0:
    df_reg["z_robusto_target"] = 0
else:
    df_reg["z_robusto_target"] = 0.6745 * (df_reg["marca_vs_nivel_cm"] - mediana) / mad

df_reg["outlier_mad_target"] = df_reg["z_robusto_target"].abs() > 3.5


df_reg["extremo_target"] = (
    df_reg["outlier_iqr_target"] |
    df_reg["outlier_mad_target"]
)

resumen_extremos = pd.DataFrame([{
    "n_saltos_regresion_completo": len(df_reg),
    "q1": q1,
    "q3": q3,
    "iqr": iqr,
    "limite_inferior_iqr": limite_inferior_iqr,
    "limite_superior_iqr": limite_superior_iqr,
    "mediana": mediana,
    "mad": mad,
    "n_outliers_iqr": int(df_reg["outlier_iqr_target"].sum()),
    "n_outliers_mad": int(df_reg["outlier_mad_target"].sum()),
    "n_extremos_final": int(df_reg["extremo_target"].sum()),
    "n_saltos_sin_extremos": int((~df_reg["extremo_target"]).sum())
}])

display(resumen_extremos)

resumen_extremos.to_csv(
    SENS_TABLAS_DIR / "resumen_deteccion_extremos_target.csv",
    index=False,
    encoding="utf-8-sig"
)

extremos_target = df_reg[df_reg["extremo_target"]].copy()

extremos_target = extremos_target.sort_values("marca_vs_nivel_cm")

columnas_mostrar_extremos = [
    c for c in [
        "video_id",
        "atleta",
        "resultado_original",
        "marca_m",
        "nivel_referencia_pc",
        "marca_vs_nivel_cm",
        "z_robusto_target",
        "outlier_iqr_target",
        "outlier_mad_target"
    ]
    if c in extremos_target.columns
]

display(extremos_target[columnas_mostrar_extremos])

extremos_target[columnas_mostrar_extremos].to_csv(
    SENS_TABLAS_DIR / "saltos_extremos_marca_vs_nivel_cm.csv",
    index=False,
    encoding="utf-8-sig"
)


plt.figure(figsize=(8, 5))
plt.hist(df_reg["marca_vs_nivel_cm"], bins=10, edgecolor="black")
plt.axvline(mediana, linestyle="--", label=f"Mediana = {mediana:.2f} cm")
plt.axvline(limite_inferior_iqr, linestyle=":", label=f"Límite inferior IQR = {limite_inferior_iqr:.2f}")
plt.axvline(limite_superior_iqr, linestyle=":", label=f"Límite superior IQR = {limite_superior_iqr:.2f}")
plt.xlabel("Marca respecto al nivel del atleta (cm)")
plt.ylabel("Número de saltos")
plt.title("Distribución del rendimiento relativo y valores extremos")
plt.legend()
plt.tight_layout()

plt.savefig(
    SENS_FIGURAS_DIR / "distribucion_target_con_limites_extremos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.figure(figsize=(10, 5))
plt.scatter(
    range(len(df_reg)),
    df_reg["marca_vs_nivel_cm"],
    label="Saltos no extremos"
)

if len(extremos_target) > 0:
    posiciones_extremos = df_reg.reset_index(drop=True).index[df_reg["extremo_target"].values]
    plt.scatter(
        posiciones_extremos,
        df_reg.reset_index(drop=True).loc[posiciones_extremos, "marca_vs_nivel_cm"],
        label="Saltos extremos"
    )

plt.axhline(mediana, linestyle="--", label="Mediana")
plt.axhline(limite_inferior_iqr, linestyle=":", label="Límite inferior IQR")
plt.axhline(limite_superior_iqr, linestyle=":", label="Límite superior IQR")
plt.xlabel("Índice del salto")
plt.ylabel("Marca respecto al nivel del atleta (cm)")
plt.title("Valores extremos en el rendimiento relativo")
plt.legend()
plt.tight_layout()

plt.savefig(
    SENS_FIGURAS_DIR / "scatter_extremos_target_marca_vs_nivel.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


def preparar_X_y_y_paquetes(df_entrada):
    y = df_entrada["marca_vs_nivel_cm"].astype(float)

    columnas_excluir = [
        "video_id",
        "id_salto",
        "competicion",
        "intento",
        "atleta",
        "id_atleta",
        "resultado_original",
        "marca_m",
        "resultado_nulo",
        "nivel_referencia_pc",
        "marca_vs_nivel_cm",
        "outlier_iqr_target",
        "outlier_mad_target",
        "z_robusto_target",
        "extremo_target",
    ]

    columnas_excluir = [c for c in columnas_excluir if c in df_entrada.columns]

    X_total = df_entrada.drop(columns=columnas_excluir)
    X_total = X_total.select_dtypes(include=[np.number]).copy()

    patrones_prohibidos = [
        "marca",
        "resultado",
        "nivel",
        "atleta",
        "video",
        "competicion",
        "intento",
        "id_salto",
        "outlier",
        "z_robusto",
        "extremo"
    ]

    columnas_sospechosas = [
        c for c in X_total.columns
        if any(p in c.lower() for p in patrones_prohibidos)
    ]

    if columnas_sospechosas:
        X_total = X_total.drop(columns=columnas_sospechosas)

    columnas = list(X_total.columns)

    def contiene(col, patrones):
        col = col.lower()
        return any(p in col for p in patrones)

    cols_angulos = [
        c for c in columnas
        if contiene(
            c,
            [
                "angulo",
                "rodilla",
                "cadera",
                "tronco",
                "inclinacion",
                "dif_",
                "diferencia"
            ]
        )
    ]

    cols_velocidades = [
        c for c in columnas
        if contiene(c, ["velocidad", "vel_"])
        and not contiene(c, ["time", "tiempo", "delta_t"])
    ]

    cols_temporales_espaciales = [
        c for c in columnas
        if contiene(
            c,
            [
                "distancia",
                "longitud",
                "tiempo",
                "time",
                "delta_t",
                "delta_x",
                "delta_y"
            ]
        )
    ]

    paquetes = {
        "angulos": list(dict.fromkeys(cols_angulos)),
        "velocidades": list(dict.fromkeys(cols_velocidades)),
        "temporales_espaciales": list(dict.fromkeys(cols_temporales_espaciales)),
        "todas_biomecanicas": list(dict.fromkeys(columnas)),
    }

    return X_total, y, paquetes


modelos = {
    "Dummy": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyRegressor())
        ]),
        "param_grid": {
            "model__strategy": ["mean", "median"]
        }
    },

    "Ridge": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(random_state=42))
        ]),
        "param_grid": {
            "model__alpha": [0.1, 1, 10, 100]
        }
    },

    "ElasticNet": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(
                max_iter=5000,
                random_state=42
            ))
        ]),
        "param_grid": {
            "model__alpha": [0.01, 0.1, 1, 10],
            "model__l1_ratio": [0.1, 0.5, 0.9]
        }
    },

    "SVR": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR())
        ]),
        "param_grid": {
            "model__kernel": ["linear", "rbf"],
            "model__C": [0.1, 1, 10],
            "model__epsilon": [1, 5, 10],
            "model__gamma": ["scale"]
        }
    },

    "Random_Forest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 3, 5],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", 1.0]
        }
    },

    "Gradient_Boosting": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [1, 3],
            "model__subsample": [0.8, 1.0]
        }
    }
}


def ejecutar_experimento_regresion(df_escenario, nombre_escenario):
    print("\n" + "=" * 90)
    print(f"ESCENARIO: {nombre_escenario}")
    print("=" * 90)

    X_total, y, paquetes_variables = preparar_X_y_y_paquetes(df_escenario)

    print("N saltos:", len(y))
    print("Media target:", y.mean())
    print("Mediana target:", y.median())
    print("STD target:", y.std())
    print("Min target:", y.min())
    print("Max target:", y.max())

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scoring = {
        "neg_mae": "neg_mean_absolute_error",
        "neg_rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }

    resultados = []
    predicciones = []

    for nombre_paquete, columnas_paquete in paquetes_variables.items():

        if len(columnas_paquete) == 0:
            continue

        X = X_total[columnas_paquete].copy()

        for nombre_modelo, config in modelos.items():
            print(f"Entrenando: {nombre_escenario} | {nombre_paquete} | {nombre_modelo}")

            grid = GridSearchCV(
                estimator=config["pipeline"],
                param_grid=config["param_grid"],
                scoring=scoring,
                refit="neg_mae",
                cv=cv,
                n_jobs=-1,
                return_train_score=True
            )

            grid.fit(X, y)

            mejor_idx = grid.best_index_
            mejor_modelo = grid.best_estimator_

            y_pred_cv = cross_val_predict(
                mejor_modelo,
                X,
                y,
                cv=cv,
                n_jobs=-1
            )

            mae = mean_absolute_error(y, y_pred_cv)
            rmse = np.sqrt(mean_squared_error(y, y_pred_cv))
            r2 = r2_score(y, y_pred_cv)

            resultados.append({
                "escenario": nombre_escenario,
                "paquete": nombre_paquete,
                "modelo": nombre_modelo,
                "n_variables": len(columnas_paquete),
                "n_saltos": len(y),
                "target_media": y.mean(),
                "target_mediana": y.median(),
                "target_std": y.std(),
                "target_min": y.min(),
                "target_max": y.max(),

                "mae_cv_pred_cm": mae,
                "rmse_cv_pred_cm": rmse,
                "r2_cv_pred": r2,

                "mae_grid_mean_cm": -grid.cv_results_["mean_test_neg_mae"][mejor_idx],
                "mae_grid_std_cm": grid.cv_results_["std_test_neg_mae"][mejor_idx],
                "rmse_grid_mean_cm": -grid.cv_results_["mean_test_neg_rmse"][mejor_idx],
                "r2_grid_mean": grid.cv_results_["mean_test_r2"][mejor_idx],

                "mejores_hiperparametros": json.dumps(grid.best_params_, ensure_ascii=False)
            })

            pred_tmp = df_escenario[[
                c for c in [
                    "video_id",
                    "atleta",
                    "resultado_original",
                    "marca_m",
                    "nivel_referencia_pc",
                    "marca_vs_nivel_cm"
                ]
                if c in df_escenario.columns
            ]].copy()

            pred_tmp["escenario"] = nombre_escenario
            pred_tmp["paquete"] = nombre_paquete
            pred_tmp["modelo"] = nombre_modelo
            pred_tmp["y_real_cm"] = y.values
            pred_tmp["y_pred_cm"] = y_pred_cv
            pred_tmp["error_cm"] = pred_tmp["y_pred_cm"] - pred_tmp["y_real_cm"]
            pred_tmp["abs_error_cm"] = pred_tmp["error_cm"].abs()

            predicciones.append(pred_tmp)

    resultados_df = pd.DataFrame(resultados)

    resultados_df = resultados_df.sort_values(
        ["mae_cv_pred_cm", "rmse_cv_pred_cm"],
        ascending=[True, True]
    ).reset_index(drop=True)

    predicciones_df = pd.concat(predicciones, ignore_index=True)

    return resultados_df, predicciones_df


df_reg_completo = df_reg.copy()
df_reg_sin_extremos = df_reg[~df_reg["extremo_target"]].copy()

resultados_completo, predicciones_completo = ejecutar_experimento_regresion(
    df_reg_completo,
    "completo"
)

resultados_sin_extremos, predicciones_sin_extremos = ejecutar_experimento_regresion(
    df_reg_sin_extremos,
    "sin_extremos_target"
)


resultados_sensibilidad = pd.concat(
    [resultados_completo, resultados_sin_extremos],
    ignore_index=True
)

resultados_sensibilidad = resultados_sensibilidad.sort_values(
    ["escenario", "mae_cv_pred_cm", "rmse_cv_pred_cm"],
    ascending=[True, True, True]
).reset_index(drop=True)

display(resultados_sensibilidad)

resultados_sensibilidad.to_csv(
    SENS_TABLAS_DIR / "resultados_sensibilidad_regresion_completo_vs_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)

predicciones_sensibilidad = pd.concat(
    [predicciones_completo, predicciones_sin_extremos],
    ignore_index=True
)

predicciones_sensibilidad.to_csv(
    SENS_TABLAS_DIR / "predicciones_sensibilidad_regresion.csv",
    index=False,
    encoding="utf-8-sig"
)


mejor_por_escenario = (
    resultados_sensibilidad
    .sort_values(
        ["escenario", "mae_cv_pred_cm", "rmse_cv_pred_cm"],
        ascending=[True, True, True]
    )
    .groupby("escenario")
    .head(1)
    .reset_index(drop=True)
)

display(mejor_por_escenario)

mejor_por_escenario.to_csv(
    SENS_TABLAS_DIR / "mejor_modelo_por_escenario_sensibilidad_regresion.csv",
    index=False,
    encoding="utf-8-sig"
)


comparacion_svr_velocidades = resultados_sensibilidad[
    (resultados_sensibilidad["paquete"] == "velocidades") &
    (resultados_sensibilidad["modelo"] == "SVR")
].copy()

display(comparacion_svr_velocidades)

comparacion_svr_velocidades.to_csv(
    SENS_TABLAS_DIR / "comparacion_svr_velocidades_completo_vs_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)


plot_df = mejor_por_escenario.sort_values("mae_cv_pred_cm", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_df["escenario"], plot_df["mae_cv_pred_cm"])
plt.xlabel("MAE medio absoluto (cm)")
plt.ylabel("Escenario")
plt.title("Mejor MAE por escenario de regresión")
plt.tight_layout()

plt.savefig(
    SENS_FIGURAS_DIR / "mejor_mae_por_escenario_sensibilidad.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


if not comparacion_svr_velocidades.empty:
    plt.figure(figsize=(8, 5))
    plt.bar(
        comparacion_svr_velocidades["escenario"],
        comparacion_svr_velocidades["mae_cv_pred_cm"]
    )
    plt.ylabel("MAE medio absoluto (cm)")
    plt.title("SVR + velocidades: completo vs sin extremos")
    plt.tight_layout()

    plt.savefig(
        SENS_FIGURAS_DIR / "svr_velocidades_mae_completo_vs_sin_extremos.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


plt.figure(figsize=(8, 5))
plt.barh(plot_df["escenario"], plot_df["r2_cv_pred"])
plt.xlabel("R²")
plt.ylabel("Escenario")
plt.title("R² del mejor modelo por escenario")
plt.tight_layout()

plt.savefig(
    SENS_FIGURAS_DIR / "r2_mejor_modelo_por_escenario_sensibilidad.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


mejor_sin_extremos = mejor_por_escenario[
    mejor_por_escenario["escenario"] == "sin_extremos_target"
].iloc[0]

pred_mejor_sin_extremos = predicciones_sensibilidad[
    (predicciones_sensibilidad["escenario"] == "sin_extremos_target") &
    (predicciones_sensibilidad["paquete"] == mejor_sin_extremos["paquete"]) &
    (predicciones_sensibilidad["modelo"] == mejor_sin_extremos["modelo"])
].copy()

pred_mejor_sin_extremos = pred_mejor_sin_extremos.sort_values(
    "abs_error_cm",
    ascending=False
)

display(pred_mejor_sin_extremos)

pred_mejor_sin_extremos.to_csv(
    SENS_TABLAS_DIR / "predicciones_mejor_modelo_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)

plt.figure(figsize=(6, 6))
plt.scatter(
    pred_mejor_sin_extremos["y_real_cm"],
    pred_mejor_sin_extremos["y_pred_cm"]
)

min_val = min(
    pred_mejor_sin_extremos["y_real_cm"].min(),
    pred_mejor_sin_extremos["y_pred_cm"].min()
)

max_val = max(
    pred_mejor_sin_extremos["y_real_cm"].max(),
    pred_mejor_sin_extremos["y_pred_cm"].max()
)

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Valor real marca_vs_nivel_cm")
plt.ylabel("Predicción marca_vs_nivel_cm")
plt.title("Real vs predicho - Mejor modelo sin extremos")
plt.tight_layout()

plt.savefig(
    SENS_FIGURAS_DIR / "real_vs_predicho_mejor_modelo_sin_extremos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\n" + "=" * 90)
print("RESUMEN ANÁLISIS DE SENSIBILIDAD")
print("=" * 90)

print("Saltos en regresión completa:", len(df_reg_completo))
print("Saltos extremos detectados:", int(df_reg_completo["extremo_target"].sum()))
print("Saltos en regresión sin extremos:", len(df_reg_sin_extremos))

print("\nMejor modelo por escenario:")
display(mejor_por_escenario)

print("\nComparación específica SVR + velocidades:")
display(comparacion_svr_velocidades)

print("\nArchivos guardados en:")
print(SENS_DIR.resolve())


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import json

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.base import clone

warnings.filterwarnings("ignore")


SALIDA_DIR = Path("resultados_revision_individual") / "eda_por_bloques_apoyo_aire_tfg_v2"
DATASETS_DIR = SALIDA_DIR / "datasets"

INTERP_DIR = SALIDA_DIR / "ml_regresion_interpretabilidad_sin_extremos"
INTERP_TABLAS_DIR = INTERP_DIR / "tablas"
INTERP_FIGURAS_DIR = INTERP_DIR / "figuras"

for carpeta in [INTERP_DIR, INTERP_TABLAS_DIR, INTERP_FIGURAS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)


if "combinado_total_FINAL_con_resultado_y_nivel" in globals():
    df_ml = combinado_total_FINAL_con_resultado_y_nivel.copy()
    print("Usando dataset desde memoria.")
else:
    ruta_df = DATASETS_DIR / "combinado_total_FINAL_con_resultado_y_nivel.csv"

    if not ruta_df.exists():
        raise FileNotFoundError(f"No encuentro el dataset en:\n{ruta_df.resolve()}")

    df_ml = pd.read_csv(ruta_df, encoding="utf-8-sig")
    print("Dataset cargado desde:")
    print(ruta_df.resolve())

print("Dimensiones dataset completo:", df_ml.shape)


df_reg = df_ml[
    (df_ml["resultado_nulo"] == 0) &
    (df_ml["marca_vs_nivel_cm"].notna())
].copy()

df_reg["marca_vs_nivel_cm"] = df_reg["marca_vs_nivel_cm"].astype(float)

print("\nSaltos válidos con target disponible:", len(df_reg))


y_target = df_reg["marca_vs_nivel_cm"]

q1 = y_target.quantile(0.25)
q3 = y_target.quantile(0.75)
iqr = q3 - q1

limite_inferior_iqr = q1 - 1.5 * iqr
limite_superior_iqr = q3 + 1.5 * iqr

df_reg["outlier_iqr_target"] = (
    (df_reg["marca_vs_nivel_cm"] < limite_inferior_iqr) |
    (df_reg["marca_vs_nivel_cm"] > limite_superior_iqr)
)

mediana = y_target.median()
mad = np.median(np.abs(y_target - mediana))

if mad == 0:
    df_reg["z_robusto_target"] = 0
else:
    df_reg["z_robusto_target"] = 0.6745 * (df_reg["marca_vs_nivel_cm"] - mediana) / mad

df_reg["outlier_mad_target"] = df_reg["z_robusto_target"].abs() > 3.5

df_reg["extremo_target"] = (
    df_reg["outlier_iqr_target"] |
    df_reg["outlier_mad_target"]
)

df_sin_extremos = df_reg[~df_reg["extremo_target"]].copy()

extremos = df_reg[df_reg["extremo_target"]].copy()

print("\nExtremos detectados:", len(extremos))
display(extremos[[
    c for c in [
        "video_id",
        "atleta",
        "resultado_original",
        "marca_m",
        "nivel_referencia_pc",
        "marca_vs_nivel_cm",
        "z_robusto_target"
    ]
    if c in extremos.columns
]])

print("\nSaltos usados en interpretabilidad sin extremos:", len(df_sin_extremos))

extremos.to_csv(
    INTERP_TABLAS_DIR / "saltos_extremos_excluidos_interpretabilidad.csv",
    index=False,
    encoding="utf-8-sig"
)


columnas_excluir = [
    "video_id",
    "id_salto",
    "competicion",
    "intento",
    "atleta",
    "id_atleta",
    "resultado_original",
    "marca_m",
    "resultado_nulo",
    "nivel_referencia_pc",
    "marca_vs_nivel_cm",
    "outlier_iqr_target",
    "outlier_mad_target",
    "z_robusto_target",
    "extremo_target",
]

columnas_excluir = [c for c in columnas_excluir if c in df_sin_extremos.columns]

X_total = df_sin_extremos.drop(columns=columnas_excluir)
X_total = X_total.select_dtypes(include=[np.number]).copy()

patrones_prohibidos = [
    "marca",
    "resultado",
    "nivel",
    "atleta",
    "video",
    "competicion",
    "intento",
    "id_salto",
    "outlier",
    "z_robusto",
    "extremo"
]

columnas_sospechosas = [
    c for c in X_total.columns
    if any(p in c.lower() for p in patrones_prohibidos)
]

if columnas_sospechosas:
    X_total = X_total.drop(columns=columnas_sospechosas)

def contiene(col, patrones):
    col = col.lower()
    return any(p in col for p in patrones)

cols_temporales_espaciales = [
    c for c in X_total.columns
    if contiene(
        c,
        [
            "distancia",
            "longitud",
            "tiempo",
            "time",
            "delta_t",
            "delta_x",
            "delta_y"
        ]
    )
]

cols_temporales_espaciales = list(dict.fromkeys(cols_temporales_espaciales))

X = X_total[cols_temporales_espaciales].copy()
y = df_sin_extremos["marca_vs_nivel_cm"].astype(float)

print("\nVariables temporales y espaciales usadas:")
print("Número de variables:", X.shape[1])
display(pd.DataFrame({"variable": X.columns}))

pd.DataFrame({"variable": X.columns}).to_csv(
    INTERP_TABLAS_DIR / "variables_temporales_espaciales_rf_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)


cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

pipeline_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(random_state=42))
])

param_grid_rf = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [None, 3, 5],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 1.0]
}

scoring = {
    "neg_mae": "neg_mean_absolute_error",
    "neg_rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

grid_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    scoring=scoring,
    refit="neg_mae",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_rf.fit(X, y)

mejor_rf = grid_rf.best_estimator_

print("\nMejores hiperparámetros Random Forest:")
print(grid_rf.best_params_)

with open(
    INTERP_TABLAS_DIR / "mejores_hiperparametros_rf_temporales_sin_extremos.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(grid_rf.best_params_, f, ensure_ascii=False, indent=4)


y_pred_cv = cross_val_predict(
    mejor_rf,
    X,
    y,
    cv=cv,
    n_jobs=-1
)

mae = mean_absolute_error(y, y_pred_cv)
rmse = np.sqrt(mean_squared_error(y, y_pred_cv))
r2 = r2_score(y, y_pred_cv)

metricas_rf = pd.DataFrame([{
    "escenario": "sin_extremos_target",
    "paquete": "temporales_espaciales",
    "modelo": "Random_Forest",
    "n_saltos": len(y),
    "n_variables": X.shape[1],
    "target_media": y.mean(),
    "target_mediana": y.median(),
    "target_std": y.std(),
    "mae_cv_pred_cm": mae,
    "rmse_cv_pred_cm": rmse,
    "r2_cv_pred": r2,
    "mejores_hiperparametros": json.dumps(grid_rf.best_params_, ensure_ascii=False)
}])

display(metricas_rf)

metricas_rf.to_csv(
    INTERP_TABLAS_DIR / "metricas_rf_temporales_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)


predicciones = df_sin_extremos[[
    c for c in [
        "video_id",
        "atleta",
        "resultado_original",
        "marca_m",
        "nivel_referencia_pc",
        "marca_vs_nivel_cm"
    ]
    if c in df_sin_extremos.columns
]].copy()

predicciones["y_real_cm"] = y.values
predicciones["y_pred_cm"] = y_pred_cv
predicciones["error_cm"] = predicciones["y_pred_cm"] - predicciones["y_real_cm"]
predicciones["abs_error_cm"] = predicciones["error_cm"].abs()

predicciones = predicciones.sort_values("abs_error_cm", ascending=False)

display(predicciones)

predicciones.to_csv(
    INTERP_TABLAS_DIR / "predicciones_rf_temporales_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)


plt.figure(figsize=(6, 6))
plt.scatter(predicciones["y_real_cm"], predicciones["y_pred_cm"])

min_val = min(predicciones["y_real_cm"].min(), predicciones["y_pred_cm"].min())
max_val = max(predicciones["y_real_cm"].max(), predicciones["y_pred_cm"].max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Valor real marca_vs_nivel_cm")
plt.ylabel("Predicción marca_vs_nivel_cm")
plt.title("Real vs predicho - RF temporales sin extremos")
plt.tight_layout()

plt.savefig(
    INTERP_FIGURAS_DIR / "real_vs_predicho_rf_temporales_sin_extremos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


plt.figure(figsize=(8, 5))
plt.hist(predicciones["error_cm"], bins=10, edgecolor="black")
plt.axvline(0, linestyle="--")
plt.xlabel("Error de predicción (cm)")
plt.ylabel("Número de saltos")
plt.title("Distribución de errores - RF temporales sin extremos")
plt.tight_layout()

plt.savefig(
    INTERP_FIGURAS_DIR / "distribucion_errores_rf_temporales_sin_extremos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


mejor_rf.fit(X, y)

rf_model = mejor_rf.named_steps["model"]

importancia_rf = pd.DataFrame({
    "variable": X.columns,
    "importancia_rf": rf_model.feature_importances_
})

importancia_rf = importancia_rf.sort_values(
    "importancia_rf",
    ascending=False
).reset_index(drop=True)

display(importancia_rf)

importancia_rf.to_csv(
    INTERP_TABLAS_DIR / "importancia_interna_random_forest.csv",
    index=False,
    encoding="utf-8-sig"
)

top_rf = importancia_rf.head(15).sort_values("importancia_rf")

plt.figure(figsize=(9, max(5, 0.35 * len(top_rf))))
plt.barh(top_rf["variable"], top_rf["importancia_rf"])
plt.xlabel("Importancia interna Random Forest")
plt.ylabel("Variable")
plt.title("Importancia interna - RF temporales sin extremos")
plt.tight_layout()

plt.savefig(
    INTERP_FIGURAS_DIR / "importancia_interna_random_forest.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


importancias_folds = []

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    modelo_fold = clone(mejor_rf)
    modelo_fold.fit(X_train, y_train)

    perm = permutation_importance(
        modelo_fold,
        X_test,
        y_test,
        scoring="neg_mean_absolute_error",
        n_repeats=50,
        random_state=42 + fold_idx,
        n_jobs=-1
    )

    fold_df = pd.DataFrame({
        "variable": X.columns,
        "aumento_mae_medio_fold": perm.importances_mean,
        "aumento_mae_std_fold": perm.importances_std,
        "fold": fold_idx
    })

    importancias_folds.append(fold_df)

importancias_folds_df = pd.concat(importancias_folds, ignore_index=True)

importancias_folds_df.to_csv(
    INTERP_TABLAS_DIR / "permutation_importance_folds_rf_temporales_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)

perm_importance = (
    importancias_folds_df
    .groupby("variable")
    .agg(
        aumento_mae_medio=("aumento_mae_medio_fold", "mean"),
        aumento_mae_std=("aumento_mae_medio_fold", "std")
    )
    .reset_index()
)

perm_importance = perm_importance.sort_values(
    "aumento_mae_medio",
    ascending=False
).reset_index(drop=True)

display(perm_importance)

perm_importance.to_csv(
    INTERP_TABLAS_DIR / "permutation_importance_rf_temporales_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)

top_perm = perm_importance.head(15).sort_values("aumento_mae_medio")

plt.figure(figsize=(9, max(5, 0.35 * len(top_perm))))
plt.barh(top_perm["variable"], top_perm["aumento_mae_medio"])
plt.xlabel("Aumento medio del MAE al permutar la variable (cm)")
plt.ylabel("Variable")
plt.title("Permutation importance - RF temporales sin extremos")
plt.tight_layout()

plt.savefig(
    INTERP_FIGURAS_DIR / "permutation_importance_rf_temporales_sin_extremos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


tabla_importancia_combinada = importancia_rf.merge(
    perm_importance,
    on="variable",
    how="outer"
)

tabla_importancia_combinada = tabla_importancia_combinada.sort_values(
    "aumento_mae_medio",
    ascending=False
).reset_index(drop=True)

display(tabla_importancia_combinada)

tabla_importancia_combinada.to_csv(
    INTERP_TABLAS_DIR / "tabla_importancia_combinada_rf_temporales_sin_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 90)
print("RESUMEN INTERPRETABILIDAD REGRESIÓN SIN EXTREMOS")
print("=" * 90)

print(f"Número de saltos utilizados: {len(y)}")
print(f"Número de variables: {X.shape[1]}")
print(f"Modelo: Random Forest")
print(f"Paquete: temporales_espaciales")
print(f"MAE: {mae:.2f} cm")
print(f"RMSE: {rmse:.2f} cm")
print(f"R²: {r2:.3f}")

print("\nVariables más importantes según Random Forest:")
display(importancia_rf.head(10))

print("\nVariables más importantes según permutation importance:")
display(perm_importance.head(10))

print("\nArchivos guardados en:")
print(INTERP_DIR.resolve())


## 9. Key takeaways

- Temporal and spatial variables contain the clearest signal for valid/foul classification.
- Absolute-distance prediction improves moderately over the baseline but remains limited.
- Relative-performance modelling improves after removing extreme target values.
- The project is exploratory: the goal is an accessible and interpretable methodology, not a production prediction system.
